# Eksperimen CODAMOSA vs CoverUp

Notebook ini menjalankan **generator CODAMOSA dan CoverUp**, kemudian mengevaluasi
test yang dihasilkan dengan evaluator bersama di `eval_metrics.py` pada suite paper:
MT (f001-f163) dan CM (425 modul). MuTAP dijalankan rekan di device lain. Ubah
`TASK_LIMIT=None` di `.env` setelah pilot berhasil untuk menjalankan semua task.

Credential hanya dibaca dari environment kernel dan tidak disimpan ke notebook.

### Kernel Python wajib

Pilih kernel **Python (.venv Teori Pengujian)** sebelum menjalankan cell. Kernel harus
menggunakan `./.venv/bin/python`, bukan `/usr/bin/python3`. Jika kernel belum muncul,
jalankan `./.venv/bin/python -m ipykernel install --user --name teori-pengujian
--display-name 'Python (.venv Teori Pengujian)'`, lalu pilih kernel tersebut.

Preflight mengirim satu request kecil ke provider, sehingga request tersebut dapat
dikenai biaya.

### Format model

Notebook ini dikonfigurasi untuk endpoint OpenAI-compatible OctaLabs
(`https://router.octalabs.id/v1`). Untuk CoverUp, `openai/` hanyalah prefix
transport LiteLLM; contoh `openai/claude-sonnet-5` dikirim sebagai
`claude-sonnet-5`. CODAMOSA mengirim bare model ID secara langsung.
Alias lama `entah/...` dinormalisasi otomatis. Setiap model dijalankan terpisah.

Jika salah satu metode gagal, notebook tidak mengarang skor. Status `BLOCKED` dan
diagnosis disimpan di `experiment_results/status.json`; metode lain tetap dicoba.


In [1]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys
import time
import tempfile
import importlib.util
import hashlib

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(dotenv_path, override=False):
        path = Path(dotenv_path)
        if not path.exists():
            return False
        for raw_line in path.read_text(encoding='utf-8').splitlines():
            line = raw_line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            key, value = key.strip(), value.strip()
            if len(value) >= 2 and value[0] == value[-1] and value[0] in (chr(39), chr(34)):
                value = value[1:-1]
            if override or key not in os.environ:
                os.environ[key] = value
        return True

ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'eval_metrics.py').exists():
    raise FileNotFoundError(
        'Buka notebook dari root repositori yang berisi src/eval_metrics.py '
        'atau sesuaikan ROOT.'
    )
sys.path.insert(0, str(ROOT / 'src'))
import eval_metrics as em

# Fail early with the actual kernel interpreter instead of producing misleading
# per-task BLOCKED messages when VS Code selected a stale Linux .venv kernel.
REQUIRED_HOST_PACKAGES = (
    'litellm', 'requests', 'dotenv', 'nbclient', 'aiolimiter',
    'numpy', 'pandas', 'matplotlib', 'mutpy',
)
missing_host_packages = [name for name in REQUIRED_HOST_PACKAGES if importlib.util.find_spec(name) is None]
if missing_host_packages:
    raise RuntimeError(
        'Dependency host tidak lengkap pada interpreter notebook ' + sys.executable
        + ': ' + ', '.join(missing_host_packages)
        + '. Pilih kernel .venv project di VS Code. Perbaiki dengan: '
        + sys.executable + ' -m pip install -r config/requirements.txt ; '
        + sys.executable + ' -m ipykernel install --user --name teori-pengujian '
        + "--display-name 'Python (.venv Teori Pengujian)'."
    )
print('Notebook interpreter:', sys.executable)
print('Host dependencies: OK (' + ', '.join(REQUIRED_HOST_PACKAGES) + ')')

load_dotenv(ROOT / 'config' / '.env', override=True)
# OctaLabs is the actual provider. Populate compatibility names only in-memory
# because LiteLLM and the CodaMOSA CLI require these conventional names.
_octa_key = (os.getenv('OCTA_API_KEY') or os.getenv('OPENAI_API_KEY')
             or os.getenv('CODAMOSA_API_KEY') or '').strip()
if _octa_key:
    os.environ['OPENAI_API_KEY'] = _octa_key
    os.environ['CODAMOSA_API_KEY'] = _octa_key
if os.getenv('OPENAI_BASE_URL') and not os.getenv('OPENAI_API_BASE'):
    os.environ['OPENAI_API_BASE'] = os.environ['OPENAI_BASE_URL']
os.environ.setdefault('LITELLM_LOCAL_MODEL_COST_MAP', 'True')

DATASETS = ['MT', 'CM']
_task_limit_raw = os.getenv('TASK_LIMIT', 'all').strip()
TASK_LIMIT = None if _task_limit_raw.lower() in {'', 'none', 'all', '0'} else int(_task_limit_raw)

def provider_model_id(model):
    model = model.strip()
    return model[len('entah/'):] if model.startswith('entah/') else model

def coverup_model_name(model):
    model = provider_model_id(model)
    return model if model.startswith('openai/') else 'openai/' + model

def codamosa_model_name(model):
    model = provider_model_id(model)
    return model[len('openai/'):] if model.startswith('openai/') else model

COVERUP_MODELS = [coverup_model_name(m) for m in os.getenv('COVERUP_MODELS', 'openai/gpt-5.6-luna').split(',') if m.strip()]
COVERUP_TIMEOUT = int(os.getenv('COVERUP_TIMEOUT', '900'))
CODAMOSA_MODELS = [codamosa_model_name(m) for m in os.getenv('CODAMOSA_MODELS', 'gpt-5.6-luna').split(',') if m.strip()]
assert COVERUP_MODELS and all(m.startswith('openai/') for m in COVERUP_MODELS), COVERUP_MODELS
assert CODAMOSA_MODELS and all(not m.startswith('openai/') for m in CODAMOSA_MODELS), CODAMOSA_MODELS
CODAMOSA_IMAGE = os.getenv('CODAMOSA_IMAGE', 'pynguin-runner:latest')
CODAMOSA_SEARCH_SECONDS = int(os.getenv('CODAMOSA_SEARCH_SECONDS', '60'))
CODAMOSA_POPULATION = int(os.getenv('CODAMOSA_POPULATION', '5'))
CODAMOSA_TEMPERATURE = float(os.getenv('CODAMOSA_TEMPERATURE', '0.8'))
COVERUP_MAX_ATTEMPTS = max(1, int(os.getenv('COVERUP_MAX_ATTEMPTS', '3')))
COVERUP_MODEL_TEMPERATURE = float(os.getenv('COVERUP_MODEL_TEMPERATURE', '0'))

# Use every CPU thread except a few reserved for desktop/navigation.
TASK_WORKERS = max(1, int(os.getenv('TASK_WORKERS', '2')))
_reserved_threads = max(1, int(os.getenv('CPU_RESERVED_THREADS', '1')))
_cpu_budget = max(1, (os.cpu_count() or 6) - _reserved_threads)
_requested_mutant_workers = max(1, int(os.getenv('EVAL_MUTANT_WORKERS', '3')))
EVAL_MUTANT_WORKERS = min(
    _requested_mutant_workers,
    max(1, _cpu_budget // TASK_WORKERS),
)
os.environ['EVAL_MUTANT_WORKERS'] = str(EVAL_MUTANT_WORKERS)
CODAMOSA_DOCKER_CPUS = max(1.0, float(os.getenv('CODAMOSA_DOCKER_CPUS', '2')))
CODAMOSA_RAM_GB_PER_WORKER = max(1.0, float(os.getenv('CODAMOSA_RAM_GB_PER_WORKER', '3')))
GENERATOR_ATTEMPTS = max(1, int(os.getenv('GENERATOR_ATTEMPTS', '3')))
GENERATOR_RETRY_SECONDS = max(0.0, float(os.getenv('GENERATOR_RETRY_SECONDS', '3')))
VERBOSE_TASK_LOG = os.getenv('VERBOSE_TASK_LOG', '0').strip().lower() in {'1', 'true', 'yes'}
print('Experiment mode: fresh luna')
print('Parallel task workers:', TASK_WORKERS)
print('Effective mutation workers:', EVAL_MUTANT_WORKERS)
print('Quality budget: CoverUp attempts=%d temp=%.2f | CODAMOSA search=%ds population=%d temp=%.2f' % (
    COVERUP_MAX_ATTEMPTS, COVERUP_MODEL_TEMPERATURE, CODAMOSA_SEARCH_SECONDS,
    CODAMOSA_POPULATION, CODAMOSA_TEMPERATURE))
print('CPU budget (threads):', _cpu_budget, 'of', os.cpu_count(), '| reserved:', _reserved_threads)

# Label shown in every table/figure. The provider only exposes a router alias
# (`auto`); the requested backbone is recorded here for the paper while
# `requested_model`/`returned_model` keep the raw provenance auditable.
BACKBONE_LABEL = os.getenv('BACKBONE_LABEL', '').strip() or None
if BACKBONE_LABEL:
    print('Backbone label:', BACKBONE_LABEL)

# Shared request budget for every provider call, including CoverUp's child
# process and the CodaMOSA container.  One file = one 50 RPM window.
RPM_GATE_STATE = ROOT / 'experiment_results' / 'rpm_gate.json'
os.environ['RPM_GATE_STATE'] = str(RPM_GATE_STATE)
os.environ.setdefault('RPM_LIMIT', '50')
print('RPM gate:', RPM_GATE_STATE, '| limit:', os.environ['RPM_LIMIT'])
# Each shard writes to its own folder so parallel instances never race on
# rows.csv/status.json/work.  Default keeps the historical location.
OUT = ROOT / os.getenv('RESULTS_DIR', 'experiment_results')

OUT.mkdir(parents=True, exist_ok=True)
import experiment_storage as _storage
store = _storage.ResultStore(OUT)
store.begin(os.getenv('RUN_LABEL', 'manual'))
print('Run candidate dir:', store.candidate)
if os.getenv('FRESH', '0').strip().lower() in {'1', 'true', 'yes'}:
    print('FRESH requested; active results are still retained and will be backed up on promote.')
TASKS = {name: em.task_ids(name, TASK_LIMIT) for name in DATASETS}
for _suite, _expected in {'MT': 163, 'CM': 425}.items():
    _actual = len(em.task_ids(_suite))
    if _actual != _expected:
        raise RuntimeError(f'Benchmark {_suite} tidak lengkap: {_actual}/{_expected}')
print('Benchmark verified: MT=163, CM=425; MuTAP excluded on this device.')
_task_filter_raw = os.getenv('TASK_FILTER', '').strip()
if _task_filter_raw:
    _wanted = set()
    for _piece in _task_filter_raw.replace(';', ',').split(','):
        _piece = _piece.strip()
        if not _piece:
            continue
        if ':' in _piece:
            _ds, _tid = _piece.split(':', 1)
            _wanted.add((_ds.strip(), _tid.strip()))
        else:
            _wanted.add((None, _piece))
    TASKS = {ds: [t for t in tids if (ds, str(t)) in _wanted or (None, str(t)) in _wanted]
             for ds, tids in TASKS.items()}
    TASKS = {ds: tids for ds, tids in TASKS.items() if tids}
    print('TASK_FILTER:', _task_filter_raw, '->', TASKS)
# Task sharding: N instances share one API key but run DISJOINT tasks.
# Instance i of N keeps positions where pos % N == i.  Duplicating the same
# prompt only burns quota and cannot finish faster.
_shard_raw = os.getenv('TASK_SHARD_COUNT', '').strip()
if _shard_raw:
    SHARD_COUNT = max(1, int(_shard_raw))
    SHARD_INDEX = max(0, int(os.getenv('TASK_SHARD_INDEX', '0') or 0)) % SHARD_COUNT
    TASKS = {ds: [t for pos, t in enumerate(tids) if pos % SHARD_COUNT == SHARD_INDEX]
             for ds, tids in TASKS.items()}
    TASKS = {ds: tids for ds, tids in TASKS.items() if tids}
    print('TASK SHARD: instance %d/%d -> %d task' % (
        SHARD_INDEX + 1, SHARD_COUNT, sum(len(v) for v in TASKS.values())))
else:
    SHARD_COUNT, SHARD_INDEX = 1, 0
RESUME = os.getenv('RESUME', '0').strip().lower() in {'1', 'true', 'yes'}

print('Root:', ROOT)
print('Tasks:', TASKS)
print('CoverUp models:', COVERUP_MODELS)
print('CODAMOSA models configured:', CODAMOSA_MODELS)
print('OPENAI_BASE_URL:', os.getenv('OPENAI_BASE_URL'))
print('LITELLM_LOCAL_MODEL_COST_MAP:', os.getenv('LITELLM_LOCAL_MODEL_COST_MAP'))


Notebook interpreter: c:\Users\adamf\Documents\Kuliah\ITS\Semester 1\Topik Dalam Pengujian Perangkat Lunak\Project\.venv\Scripts\python.exe
Host dependencies: OK (litellm, requests, dotenv, nbclient, aiolimiter, numpy, pandas, matplotlib, mutpy)
Experiment mode: fresh luna
Parallel task workers: 1
Effective mutation workers: 2
Quality budget: CoverUp attempts=10 temp=1.00 | CODAMOSA search=600s population=50 temp=1.00
CPU budget (threads): 14 of 16 | reserved: 2
Backbone label: GPT-5.6 Sol
RPM gate: C:\Users\adamf\Documents\Kuliah\ITS\Semester 1\Topik Dalam Pengujian Perangkat Lunak\Project\experiment_results\rpm_gate.json | limit: 50
Run candidate dir: C:\Users\adamf\Documents\Kuliah\ITS\Semester 1\Topik Dalam Pengujian Perangkat Lunak\Project\experiment_results\runs\20261019T214958Z_srbyte-auto-full-budget
Benchmark verified: MT=163, CM=425; MuTAP excluded on this device.
Root: C:\Users\adamf\Documents\Kuliah\ITS\Semester 1\Topik Dalam Pengujian Perangkat Lunak\Project
Tasks: {'MT': 

## Kesiapan Device & Lingkungan

Cell berikut memeriksa CPU, RAM, disk, paket Python, dan Docker **sebelum** eksperimen membakar token atau waktu. Bila PC kuat, beban dapat naik otomatis (`AUTO_SCALE=1`); bila kurang, beban turun otomatis atau berhenti dengan status BLOCKED. RAM dan CPU tidak bisa diunduh. Selisih jam lokal vs provider juga diperiksa. Paket Python hanya diinstal otomatis bila `AUTO_INSTALL=1` di `config/.env`.


In [2]:
# --- Device & Environment Readiness --------------------------------------
# Cek hardware dan kebutuhan SEBELUM eksperimen membakar token/waktu.
# - Software hilang  -> coba install otomatis bila AUTO_INSTALL=1.
# - Hardware kurang  -> turunkan beban (TASK_WORKERS) otomatis; bila tetap
#                       tidak muat -> BLOCKED (berhenti).
# - RAM/CPU tidak bisa diunduh: keduanya hardware fisik.
import platform as _platform

_READINESS_NOTES = []
_READINESS_BLOCKERS = []


def _readiness_note(text):
    _READINESS_NOTES.append(text)
    print(text)


def _readiness_block(text):
    _READINESS_BLOCKERS.append(text)
    print('BLOCKED: ' + text)


def _readiness_gb(n_bytes):
    return round(n_bytes / (1024 ** 3), 2)


def _readiness_host_resources():
    cpu = os.cpu_count() or 1
    ram_total, ram_avail = None, None
    try:
        import psutil as _psutil
        mem = _psutil.virtual_memory()
        ram_total, ram_avail = mem.total, mem.available
    except Exception:
        try:
            info = {}
            for raw in Path('/proc/meminfo').read_text().splitlines():
                key, _, val = raw.partition(':')
                info[key.strip()] = int(val.strip().split()[0]) * 1024
            ram_total = info.get('MemTotal')
            ram_avail = info.get('MemAvailable', info.get('MemFree'))
        except Exception:
            pass
    disk_free = shutil.disk_usage(str(ROOT)).free
    return {'cpu': cpu, 'ram_total': ram_total, 'ram_avail': ram_avail,
            'disk_free': disk_free,
            'os': _platform.system() + ' ' + _platform.release(),
            'python': _platform.python_version()}


# --- 0. Deteksi WSL vs Linux native ---------------------------------------
# WSL terlihat seperti Linux, tetapi punya jebakan khas: RAM/CPU dibatasi
# file .wslconfig, Docker jalan lewat Docker Desktop Windows, filesystem
# /mnt lambat, dan PC bisa sleep.
def _readiness_detect_wsl():
    rel = (_platform.release() or '').lower()
    if 'microsoft' in rel or 'wsl' in rel:
        try:
            ver = Path('/proc/version').read_text().lower()
            return ('wsl2' if 'wsl2' in ver else 'wsl1') if 'microsoft' in ver else 'wsl'
        except Exception:
            return 'wsl'
    return 'native'


_ENV_KIND = _readiness_detect_wsl()
_IS_WSL = _ENV_KIND != 'native'
_readiness_note('Lingkungan: ' + ('WSL (' + _ENV_KIND + ')' if _IS_WSL else 'Linux native') + '.')
if _IS_WSL:
    _readiness_note('Mode WSL: batas RAM/CPU mengikuti konfigurasi WSL (.wslconfig), '
                    'bukan spek fisik PC. Docker harus via Docker Desktop Windows.')
# --- 0b. Batas WSL: yang TERLIHAT belum tentu yang DIDAPAT ------------------
# File .wslconfig di Windows bisa membatasi lebih kecil dari /proc.
def _readiness_wsl_limits():
    info = {}
    for cand in (Path('/mnt/c/Users'), Path('/mnt/d/Users')):
        try:
            if cand.is_dir():
                for user_dir in cand.iterdir():
                    cfg = user_dir / '.wslconfig'
                    if cfg.is_file():
                        try:
                            info[str(cfg)] = cfg.read_text(errors='replace')[:800]
                        except Exception:
                            pass
        except Exception:
            pass
    return info


if _IS_WSL:
    _wsl_cfg = _readiness_wsl_limits()
    if _wsl_cfg:
        for _p, _t in _wsl_cfg.items():
            _readiness_note('Terbaca .wslconfig di ' + _p + ' (batas efektif WSL):')
            for _line in _t.splitlines():
                print('    ' + _line.strip())
    else:
        _readiness_note('Tidak menemukan .wslconfig (default WSL: 50% RAM fisik, '
                        'semua CPU). Batas efektif = yang terlihat di /proc.')
    if _ENV_KIND == 'wsl1':
        _readiness_block('Terdeteksi WSL1: tidak ada kernel Linux penuh sehingga '
                         'Docker Engine tidak bisa berjalan. Upgrade ke WSL2 '
                         '(wsl --set-version <distro> 2) di PowerShell admin.')
# --- 0c. Adaptasi otomatis per lingkungan ----------------------------------
# Aturan: tidak pernah menaikkan beban di atas permintaan user; hanya
# menurunkan bila lingkungan memaksa; setiap perubahan dicatat.
_WS_ENV = os.getenv('WSLENV', '')
if _IS_WSL:
    if shutil.which('docker') and not os.getenv('DOCKER_HOST'):
        try:
            _t = subprocess.run(['docker', 'info', '--format', '{{.ServerVersion}}'],
                                capture_output=True, text=True, encoding='utf-8',
                                errors='replace', timeout=20)
            if _t.returncode and CODAMOSA_MODELS:
                _readiness_note('Docker di WSL tak terjangkau via socket default. '
                                'Aktifkan integrasi distro ini di Docker Desktop '
                                'Windows (Settings > Resources > WSL Integration).')
        except Exception:
            pass
    try:
        if str(ROOT).startswith('/mnt/'):
            _readiness_note('PERINGATAN: repo ada di ' + str(ROOT) + ' (filesystem '
                            'Windows via /mnt). I/O work/ bisa 2-5x lebih lambat. '
                            'Disarankan clone ulang ke ~/ (home WSL) untuk run 24 jam.')
    except Exception:
        pass
    _readiness_note('PERINGATAN: pastikan Windows tidak sleep selama run 24 jam '
                    '(Settings > System > Power, sleep=Never saat dicolok).')


_RES = _readiness_host_resources()
_readiness_note('Host: ' + str(_RES['os']) + ' | Python ' + str(_RES['python']))
_readiness_note('CPU threads: ' + str(_RES['cpu']))
_readiness_note('RAM total: ' + (str(_readiness_gb(_RES['ram_total'])) + ' GB'
                                 if _RES['ram_total'] else 'tidak terdeteksi'))
_readiness_note('RAM tersedia: ' + (str(_readiness_gb(_RES['ram_avail'])) + ' GB'
                                    if _RES['ram_avail'] else 'tidak terdeteksi'))
_readiness_note('Disk bebas (repo): ' + str(_readiness_gb(_RES['disk_free'])) + ' GB')

# --- Kebutuhan yang diturunkan dari konfigurasi aktual ---
_NEED_TASK_WORKERS = max(1, int(os.getenv('TASK_WORKERS', '2')))
_NEED_MUTANT_PER_TASK = max(1, int(os.getenv('EVAL_MUTANT_WORKERS', '1')))
_NEED_DOCKER_CPUS = max(1.0, float(os.getenv('CODAMOSA_DOCKER_CPUS', '2')))
_NEED_MODELS = len(COVERUP_MODELS) + len(CODAMOSA_MODELS)
_NEED_TASKS = sum(len(v) for v in TASKS.values())
_GB_PER_CODAMOSA = CODAMOSA_RAM_GB_PER_WORKER
_GB_DESKTOP_HEADROOM = 2.0

print('Kebutuhan rencana: workers=%d mutant/task=%d docker_cpus=%.1f models=%d tasks=%d' % (
    _NEED_TASK_WORKERS, _NEED_MUTANT_PER_TASK, _NEED_DOCKER_CPUS,
    _NEED_MODELS, _NEED_TASKS))

# --- 0d. Auto scale-up bila PC kuat (tetap sisakan thread remote/navigasi) --
# Aturan: beban tidak pernah melebihi min(CONFIG_MAX_LOAD, budget aman).
# - AUTO_SCALE=1      : izinkan naik bila CPU & RAM memungkinkan.
# - Tunjangan remote : 2 thread selalu disisakan (navigasi + koneksi SSH/VSCode remote).
# - Batas keras      : CONFIG_MAX_LOAD membatasi worker meski PC kuat.
_RESERVED_THREADS = max(2, int(os.getenv('CPU_RESERVED_THREADS', '2')))
_AUTO_SCALE = os.getenv('AUTO_SCALE', '0').strip().lower() in {'1', 'true', 'yes'}
_CONFIG_MAX_LOAD = max(1, int(os.getenv('CONFIG_MAX_LOAD', '8')))

# Gate RAM: hanya informatif. Tidak lagi memblokir eksperimen;
# set RAM_MIN_GB bila tetap ingin gate aktif kembali.
_RAM_MIN_GB = max(0.0, float(os.getenv('RAM_MIN_GB', '0')))
_ram_avail_gb = _readiness_gb(_RES['ram_avail']) if _RES.get('ram_avail') else None
if _ram_avail_gb is not None and _RAM_MIN_GB > 0 and _ram_avail_gb <= _RAM_MIN_GB:
    _readiness_note(
        'RAM tersedia %.2f GB (<= %.0f GB): di bawah ambang RAM_MIN_GB, '
        'eksperimen tetap dijalankan.'
        % (_ram_avail_gb, _RAM_MIN_GB))
elif _ram_avail_gb is not None:
    _readiness_note('RAM gate: OK (%.2f GB tersedia, %.0f GB disisakan untuk OS).'
                    % (_ram_avail_gb, _GB_DESKTOP_HEADROOM))
else:
    _readiness_note('RAM gate: tidak terdeteksi (diasumsikan cukup).')

if _AUTO_SCALE:
    # Account for one generator/container core plus mutation workers per task.
    _safe_workers = max(1, (_RES['cpu'] - _RESERVED_THREADS) // (1 + _NEED_MUTANT_PER_TASK))
    if _RES.get('ram_avail'):
        _safe_workers = max(1, min(_safe_workers,
            int((_readiness_gb(_RES['ram_avail']) - _GB_DESKTOP_HEADROOM) // _GB_PER_CODAMOSA)))
    _safe_workers = min(_safe_workers, _CONFIG_MAX_LOAD)
    if TASK_WORKERS < _safe_workers:
        _readiness_note('Auto scale-up: TASK_WORKERS %d -> %d (PC kuat; %d thread disisakan untuk remote/navigasi).' % (
            TASK_WORKERS, _safe_workers, _RESERVED_THREADS))
        TASK_WORKERS = _safe_workers
        os.environ['TASK_WORKERS'] = str(TASK_WORKERS)
        _NEED_TASK_WORKERS = TASK_WORKERS
    else:
        _readiness_note('Auto scale-up: tidak perlu (TASK_WORKERS=%d sudah >= batas aman %d).' % (
            TASK_WORKERS, _safe_workers))
else:
    _readiness_note('Auto scale-up: nonaktif (AUTO_SCALE=0). Set 1 bila ingin beban naik otomatis di PC kuat.')


# --- 0e. Cek selisih jam lokal vs jam provider (clock drift) -----------------
# Jam meleset merusak timestamp manifest, log perbandingan, dan backoff 429.
# Sumber kebenaran: header HTTP Date dari endpoint provider (tanpa biaya).
_CLOCK_WARN_S = max(30, int(os.getenv('CLOCK_WARN_SECONDS', '120')))
_CLOCK_BLOCK_S = max(_CLOCK_WARN_S, int(os.getenv('CLOCK_BLOCK_SECONDS', '900')))
def _readiness_clock_skew():
    import email.utils as _eu
    import urllib.request as _ur
    base = (os.getenv('CODAMOSA_BASE_URL') or os.getenv('OPENAI_BASE_URL')
            or 'https://router.octalabs.id/v1').strip().rstrip('/')
    try:
        req = _ur.Request(base.rstrip('/') + '/models', method='GET',
                            headers={'Range': 'bytes=0-0'})
        key = (os.getenv('OCTA_API_KEY') or os.getenv('CODAMOSA_API_KEY')
               or os.getenv('OPENAI_API_KEY') or '').strip()
        if key:
            req.add_header('Authorization', 'Bearer ' + key)
        with _ur.urlopen(req, timeout=20) as resp:
            date_hdr = resp.headers.get('Date')
        if not date_hdr:
            return None, 'tanpa header Date'
        srv = _eu.parsedate_to_datetime(date_hdr)
        now = __import__('datetime').datetime.now(__import__('datetime').timezone.utc)
        return abs((now - srv).total_seconds()), None
    except Exception as exc:
        return None, '%s: %s' % (type(exc).__name__, exc)
_skew, _skew_err = _readiness_clock_skew()
if _skew is None:
    _readiness_note('Jam: tidak dapat dibaca dari provider (%s); lanjut.' % _skew_err)
elif _skew > _CLOCK_BLOCK_S:
    _readiness_block('Selisih jam lokal vs provider %.0f dtk (> %d dtk). Jalankan sudo hwclock -s lalu ulangi.' % (_skew, _CLOCK_BLOCK_S))
else:
    _readiness_note('Jam: selisih %.0f dtk vs provider (%s).' % (
        _skew, 'OK' if _skew <= _CLOCK_WARN_S else 'peringatan: sinkronkan dengan sudo hwclock -s'))


# --- 1. CPU: sisakan minimal 2 thread untuk OS/remote/navigation ---
_cpu_worker_budget = _RES['cpu'] - _RESERVED_THREADS
if _cpu_worker_budget < 1:
    _readiness_block(
        'CPU hanya %d thread; minimal perlu %d thread agar %d thread tetap tersedia untuk OS.'
        % (_RES['cpu'], _RESERVED_THREADS + 1, _RESERVED_THREADS))
elif _NEED_TASK_WORKERS > _cpu_worker_budget:
    _old = _NEED_TASK_WORKERS
    _NEED_TASK_WORKERS = _cpu_worker_budget
    TASK_WORKERS = _NEED_TASK_WORKERS
    os.environ['TASK_WORKERS'] = str(TASK_WORKERS)
    _readiness_note('Auto-tune: TASK_WORKERS %d -> %d; %d dari %d thread disisakan untuk OS.' % (
        _old, TASK_WORKERS, _RESERVED_THREADS, _RES['cpu']))
else:
    _readiness_note('CPU: OK (%d worker; %d dari %d thread disisakan untuk OS).' % (
        _NEED_TASK_WORKERS, _RESERVED_THREADS, _RES['cpu']))

# --- 2. RAM: auto-tune, bukan unduh (RAM adalah hardware) ---
_need_ram = _NEED_TASK_WORKERS * _GB_PER_CODAMOSA + _GB_DESKTOP_HEADROOM
if _RES['ram_avail'] is not None:
    if _readiness_gb(_RES['ram_avail']) < _need_ram and _NEED_TASK_WORKERS > 1:
        _old = _NEED_TASK_WORKERS
        while _NEED_TASK_WORKERS > 1 and _readiness_gb(_RES['ram_avail']) < (
                _NEED_TASK_WORKERS * _GB_PER_CODAMOSA + _GB_DESKTOP_HEADROOM):
            _NEED_TASK_WORKERS -= 1
        TASK_WORKERS = _NEED_TASK_WORKERS
        os.environ['TASK_WORKERS'] = str(TASK_WORKERS)
        _readiness_note('Auto-tune: TASK_WORKERS %d -> %d agar muat di RAM tersedia.' % (
            _old, TASK_WORKERS))
    _need_ram = _NEED_TASK_WORKERS * _GB_PER_CODAMOSA + _GB_DESKTOP_HEADROOM
    if _readiness_gb(_RES['ram_avail']) >= _need_ram:
        _readiness_note('RAM: OK (butuh ~%.1f GB, tersedia %.2f GB).' % (
            _need_ram, _readiness_gb(_RES['ram_avail'])))
    else:
        _readiness_note('RAM: PERINGATAN (butuh ~%.1f GB, tersedia %.2f GB); '
                        'RAM tidak bisa diunduh; lanjut dengan TASK_WORKERS=1.' % (
                            _need_ram, _readiness_gb(_RES['ram_avail'])))
else:
    _readiness_note('RAM: tidak terdeteksi (lanjut dengan asumsi cukup).')

# --- 3. Paket Python: install otomatis hanya bila diizinkan ---
_AUTO_INSTALL = os.getenv('AUTO_INSTALL', '0').strip().lower() in {'1', 'true', 'yes'}
_missing = [p for p in REQUIRED_HOST_PACKAGES if importlib.util.find_spec(p) is None]
if _missing and _AUTO_INSTALL:
    _readiness_note('AUTO_INSTALL=1: menginstal ' + ', '.join(_missing) + ' ...')
    try:
        _proc = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', *_missing],
            capture_output=True, text=True, encoding='utf-8',
            errors='replace', timeout=600,
        )
        _missing = [p for p in _missing if importlib.util.find_spec(p) is None]
        if _proc.returncode == 0 and not _missing:
            _readiness_note('Paket berhasil diinstal otomatis.')
        else:
            _readiness_block('pip install gagal; paket hilang: ' + ', '.join(_missing))
    except Exception as _exc:
        _readiness_block('pip install gagal: %s: %s' % (type(_exc).__name__, _exc))
elif _missing:
    _readiness_block('Paket hilang: ' + ', '.join(_missing) +
                     '. Set AUTO_INSTALL=1 untuk install otomatis atau install manual.')
else:
    _readiness_note('Paket Python: OK.')

# --- 4. Docker: cek CLI/daemon di sini, build image tetap di cell khusus ---
if CODAMOSA_MODELS:
    if not shutil.which('docker'):
        _readiness_block('Docker CLI tidak ditemukan padahal CODAMOSA_MODELS terisi.')
    else:
        try:
            _d = subprocess.run(['docker', 'info', '--format', '{{.ServerVersion}}'],
                                capture_output=True, text=True, encoding='utf-8',
                                errors='replace', timeout=20)
            if _d.returncode:
                _readiness_block('Docker daemon tidak dapat diakses. Start daemon dulu.')
            else:
                _readiness_note('Docker daemon: OK (v' + (_d.stdout or '').strip() + ').')
        except Exception as _exc:
            _readiness_block('Pemeriksaan Docker gagal: %s' % _exc)
else:
    _readiness_note('Docker: dilewati (tidak ada model CODAMOSA).')

# --- Keputusan akhir: lanjut atau berhenti SEBELUM bakar token ---
if _READINESS_BLOCKERS:
    raise RuntimeError('READINESS BLOCKED (%d masalah):\n- %s' % (
        len(_READINESS_BLOCKERS), '\n- '.join(_READINESS_BLOCKERS)))
_readiness_note('READINESS: OK (%d catatan). Lanjut ke eksperimen.' % len(_READINESS_NOTES))


Lingkungan: Linux native.
Host: Windows 11 | Python 3.14.7
CPU threads: 16
RAM total: 15.41 GB
RAM tersedia: 4.05 GB
Disk bebas (repo): 638.2 GB
Kebutuhan rencana: workers=1 mutant/task=2 docker_cpus=4.0 models=2 tasks=588
RAM gate: OK (4.05 GB tersedia, 2 GB disisakan untuk OS).
Auto scale-up: tidak perlu (TASK_WORKERS=1 sudah >= batas aman 1).
Jam: selisih 0 dtk vs provider (OK).
CPU: OK (1 worker; 2 dari 16 thread disisakan untuk OS).
RAM: PERINGATAN (butuh ~5.0 GB, tersedia 4.05 GB); RAM tidak bisa diunduh; lanjut dengan TASK_WORKERS=1.
Paket Python: OK.
Docker daemon: OK (v29.5.3).
READINESS: OK (13 catatan). Lanjut ke eksperimen.


In [3]:
# --- Model discovery & capability probe --------------------------------
# Ask the provider which models exist right now and poke each with a 1-token
# request. Optionally adopt EVERY available model for this run. Never blocks.
import model_discovery as _md

_MODELS_PATH = (os.getenv("CODAMOSA_COMPLETIONS_PATH") or "/chat/completions").strip()
_PROBE_ALL = False  # no completion probes: save tokens
_BASE = (os.getenv("CODAMOSA_BASE_URL") or os.getenv("OPENAI_BASE_URL") or "https://router.octalabs.id/v1").strip().rstrip("/")
_KEY = os.getenv("OCTA_API_KEY") or os.getenv("CODAMOSA_API_KEY") or os.getenv("OPENAI_API_KEY", "")

_configured_models = []
for _m in (os.getenv("COVERUP_MODELS", "") + "," + os.getenv("CODAMOSA_MODELS", "")).split(","):
    _m = _m.strip()
    if not _m:
        continue
    _m = _m.split("/", 1)[1] if _m.startswith("openai/") else _m
    if _m not in _configured_models:
        _configured_models.append(_m)

# Only GPT and Claude families are used in this experiment.
_FAMILY_RE = r"^gpt-5\.6-luna$"
_FAMILY_FILTER = os.getenv("MODEL_DISCOVERY_FAMILY_FILTER", _FAMILY_RE)


def _keep_model(name):
    if not _FAMILY_FILTER:
        return True
    import re as _re
    return _re.search(_FAMILY_FILTER, name, _re.IGNORECASE) is not None


_models_report = _md.discover_and_probe(
    _BASE, _KEY, completions_path=_MODELS_PATH, probe=_PROBE_ALL, timeout=90,
    include=_FAMILY_FILTER,
    workers=int(os.getenv("MODEL_DISCOVERY_WORKERS", "8")),
)
_models_report.notes.append("Configured models: " + ", ".join(_configured_models))

_USE_ALL = os.getenv("MODEL_DISCOVERY_USE_ALL", "1") not in ("0", "false", "False", "")
if _USE_ALL and _models_report.available_models:
    _max_models = int(os.getenv("MODEL_DISCOVERY_MAX_MODELS", "0") or "0")
    _selected = [m for m in _models_report.available_models if _keep_model(m)]
    _dropped = [m for m in _models_report.available_models if not _keep_model(m)]
    if _dropped:
        _models_report.notes.append(
            "Filtered out (bukan GPT/Claude): " + ", ".join(_dropped))
        print("FILTERED OUT (non GPT/Claude):", len(_dropped))
    if _max_models > 0:
        _selected = _selected[:_max_models]
    os.environ["COVERUP_MODELS"] = ",".join("openai/" + m for m in _selected)
    os.environ["CODAMOSA_MODELS"] = ",".join(_selected)
    COVERUP_MODELS = [coverup_model_name(m) for m in os.getenv("COVERUP_MODELS").split(",") if m.strip()]
    CODAMOSA_MODELS = [codamosa_model_name(m) for m in os.getenv("CODAMOSA_MODELS").split(",") if m.strip()]
    _models_report.notes.append("Using GPT/Claude models (cap=" + str(_max_models or "none") + "): " + ", ".join(_selected))
    print("USING GPT/CLAUDE MODELS:", len(_selected), _selected)

_discovery_dir = OUT / "model_discovery"
_md.write_report(_models_report, _discovery_dir / "model_discovery.json")
(_discovery_dir / "model_discovery.md").write_text(_md.render_markdown(_models_report), encoding="utf-8")

print("Run time (UTC):", _models_report.timestamp_utc)
print("Models listed:", len(_models_report.listed_models))
print("Available now:", len(_models_report.available_models))
print("Unavailable  :", len(_models_report.unavailable_models))
_unusable = [m for m in _configured_models if m not in _models_report.available_models]
print("WARNING: configured models NOT available:" if _unusable else "All configured models available.", _unusable or "")
print("Effective models:", len(COVERUP_MODELS), [m.split("/",1)[-1] for m in COVERUP_MODELS])
print("Report:", _discovery_dir / "model_discovery.md")


Run time (UTC): 2026-10-19T21:49:59Z
Models listed: 1
Available now: 1
Unavailable  : 0
All configured models available. 
Effective models: 1 ['auto']
Report: C:\Users\adamf\Documents\Kuliah\ITS\Semester 1\Topik Dalam Pengujian Perangkat Lunak\Project\experiment_results\model_discovery\model_discovery.md


## Preflight

Preflight memisahkan masalah credential, model, endpoint, dependency, dan Docker.
API key tidak pernah dicetak. Error provider disaring dari key yang diketahui.


In [4]:
def redact(value):
    text = str(value)
    for secret in (os.getenv("OCTA_API_KEY"), os.getenv("OPENAI_API_KEY"), os.getenv("CODAMOSA_API_KEY")):
        if secret:
            text = text.replace(secret, "[REDACTED_API_KEY]")
    return text[:2500]


def explain_error(exc):
    detail = redact(f"{type(exc).__name__}: {exc}")
    lower = detail.lower()
    if "insufficient_quota" in lower or "credit_balance" in lower:
        action = "Key dikenali tetapi saldo/kredit habis. Periksa billing/kuota provider."
    elif "401" in lower or "unauthorized" in lower or "invalid api key" in lower:
        action = "Credential ditolak. Periksa key, project/provider, dan environment kernel."
    elif "403" in lower or "forbidden" in lower:
        action = "Credential tidak memiliki izin untuk model atau endpoint tersebut."
    elif "404" in lower or "not found" in lower or "model_not_found" in lower:
        action = "Model/route tidak ditemukan. Periksa nama model dan URL endpoint."
    elif "429" in lower or "rate limit" in lower:
        action = "Rate limit tercapai. Kurangi request/paralelisme atau tunggu cooldown."
    elif "405" in lower or "method not allowed" in lower:
        action = "Endpoint tidak menerima method/payload ini. Pastikan path chat-completions benar."
    elif ("timeout" in lower or "connection" in lower or "network" in lower
          or "name resolution" in lower or "failed to resolve" in lower):
        action = "Endpoint tidak terjangkau/timeout. Periksa jaringan, DNS Docker, URL, dan TLS."
    elif "upstream service" in lower or "service unavailable" in lower:
        action = "Provider upstream gagal sementara. Notebook akan retry; jika berulang, tunggu cooldown atau gunakan model lain."
    else:
        action = "Periksa detail provider, model, endpoint, credential, dan dependency."
    return detail + chr(10) + "Langkah berikutnya: " + action

def _rpm_limit():
    try:
        return max(1, int(os.getenv("RPM_LIMIT", "50")))
    except ValueError:
        return 50


def _throttle(min_interval_s):
    """Serialise probes so we never exceed the provider RPM budget."""
    last = getattr(_throttle, "_last", 0.0)
    now = time.monotonic()
    wait = min_interval_s - (now - last)
    if wait > 0:
        time.sleep(wait)
        now = time.monotonic()
    _throttle._last = now


def _is_rate_limit_error(exc):
    text = str(exc).lower()
    return ("429" in text or "rate limit" in text or "too many requests" in text
            or "rate_limit" in text or "ratelimiterror" in text)


def _is_transient_probe_error(exc):
    text = str(exc).lower()
    markers = ('429', 'rate limit', 'too many requests', 'timeout', 'timed out',
               'connection', '502', '503', '504', 'service unavailable',
               'upstream service', 'upstream_error', 'temporary failure', 'eof')
    return any(marker in text for marker in markers)


def _probe_with_backoff(call, attempts, base_wait):
    """Retry transient provider/network probe failures with linear backoff."""
    last = None
    for attempt in range(1, attempts + 1):
        try:
            return call(), None
        except Exception as exc:
            last = exc
            if not _is_transient_probe_error(exc) or attempt == attempts:
                return None, exc
            time.sleep(base_wait * attempt)
    return None, last


def coverup_preflight(models):
    """Validate local CoverUp configuration without calling the model."""
    verdict = {model: True for model in models}
    if not (os.getenv("OCTA_API_KEY") or os.getenv("OPENAI_API_KEY")):
        return verdict, "BLOCKED: OCTA_API_KEY tidak ditemukan.", [], []
    if not models:
        return verdict, "BLOCKED: COVERUP_MODELS kosong.", [], []
    try:
        import litellm  # noqa: F401
    except ImportError as exc:
        return verdict, "BLOCKED: paket litellm tidak tersedia." + chr(10) + explain_error(exc), [], []
    return verdict, "SKIP: probe model CoverUp dinonaktifkan; langsung dicoba per-task.", list(models), []


def codamosa_preflight(models):
    """Validate local CODAMOSA configuration without calling the model."""
    verdict = {model: True for model in models}
    key = os.getenv("OCTA_API_KEY") or os.getenv("CODAMOSA_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not models:
        return verdict, "BLOCKED: CODAMOSA_MODELS kosong.", [], []
    if not key:
        return verdict, "BLOCKED: CODAMOSA_API_KEY atau OPENAI_API_KEY tidak ada.", [], []
    try:
        import requests  # noqa: F401
    except ImportError as exc:
        return verdict, "BLOCKED: paket requests tidak tersedia." + chr(10) + explain_error(exc), [], []
    return verdict, "SKIP: probe model CODAMOSA dinonaktifkan; langsung dicoba per-task.", list(models), []


def docker_preflight():
    if not shutil.which("docker"):
        return False, "BLOCKED: Docker CLI tidak ditemukan. Install Docker Engine/CLI."
    try:
        result = subprocess.run(
            ["docker", "info", "--format", "{{.ServerVersion}}"],
            capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=20,
        )
        if result.returncode:
            return False, ("BLOCKED: Docker daemon tidak dapat diakses: "
                           + redact((result.stderr or "") + (result.stdout or ""))
                           + ". Start daemon dan pastikan user memiliki izin Docker.")
        image = subprocess.run(["docker", "image", "inspect", CODAMOSA_IMAGE], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=20)
        if image.returncode:
            return True, "OK: Docker daemon aktif; DNS container diperiksa setelah image dibangun."
        host = os.getenv("CODAMOSA_BASE_URL", "https://router.octalabs.id/v1").split("//", 1)[-1].split("/", 1)[0].split(":", 1)[0]
        dns = subprocess.run(
            ["docker", "run", "--rm", "--entrypoint", "python", CODAMOSA_IMAGE,
             "-c", "import socket,sys; socket.getaddrinfo(sys.argv[1], 443); print('DNS OK')", host],
            capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=30,
        )
        if dns.returncode:
            return False, ("BLOCKED: container CODAMOSA tidak dapat resolve endpoint " + host + ": "
                           + redact((dns.stderr or "") + (dns.stdout or ""))
                           + ". Perbaiki DNS Docker lalu ulangi preflight.")
        return True, "OK: Docker daemon aktif (" + (result.stdout or "").strip() + ") dan DNS container berfungsi."
    except Exception as exc:
        return False, "BLOCKED: pemeriksaan Docker/DNS gagal." + chr(10) + explain_error(exc)


coverup_verdict, coverup_reason, ACTIVE_COVERUP_MODELS, coverup_errors = coverup_preflight(COVERUP_MODELS)
coverup_ok = bool(ACTIVE_COVERUP_MODELS)
codamosa_verdict, codamosa_reason, ACTIVE_CODAMOSA_MODELS, codamosa_errors = codamosa_preflight(CODAMOSA_MODELS)
codamosa_ok = bool(ACTIVE_CODAMOSA_MODELS)
docker_ok, docker_reason = docker_preflight()
if codamosa_ok and not docker_ok:
    codamosa_ok = False
    codamosa_reason += chr(10) + docker_reason

print("CoverUp:", coverup_reason)
print("CODAMOSA:", codamosa_reason)
print("Docker:", docker_reason)
print("CoverUp models available:", len(ACTIVE_COVERUP_MODELS), ACTIVE_COVERUP_MODELS)
if coverup_errors:
    print("CoverUp models skipped (transient):" + chr(10) + (chr(10) + "---" + chr(10)).join(coverup_errors[:5]))
print("CODAMOSA models available:", len(ACTIVE_CODAMOSA_MODELS), ACTIVE_CODAMOSA_MODELS)
if codamosa_errors:
    print("CODAMOSA models skipped (transient):" + chr(10) + (chr(10) + "---" + chr(10)).join(codamosa_errors[:5]))


CoverUp: SKIP: probe model CoverUp dinonaktifkan; langsung dicoba per-task.
CODAMOSA: SKIP: probe model CODAMOSA dinonaktifkan; langsung dicoba per-task.
Docker: OK: Docker daemon aktif (29.5.3) dan DNS container berfungsi.
CoverUp models available: 1 ['openai/auto']
CODAMOSA models available: 1 ['auto']


In [5]:
def _codamosa_source_sha():
    digest = hashlib.sha256()
    for candidate in sorted((ROOT / "codamosa").rglob("*")):
        if candidate.is_file() and candidate.name != ".DS_Store" and "__pycache__" not in candidate.parts:
            digest.update(str(candidate.relative_to(ROOT / "codamosa")).encode("utf-8"))
            digest.update(candidate.read_bytes())
    return digest.hexdigest()


def ensure_codamosa_image():
    source_sha = _codamosa_source_sha()
    check = subprocess.run(
        ["docker", "image", "inspect", CODAMOSA_IMAGE,
         "--format", "{{index .Config.Labels \"org.codamosa.source-sha\"}}"],
        capture_output=True,
        text=True, encoding="utf-8", errors="replace",
    )
    if check.returncode == 0 and (check.stdout or "").strip() == source_sha:
        return True, "Menggunakan image yang sinkron dengan source: " + CODAMOSA_IMAGE
    if check.returncode == 0:
        print("Image CODAMOSA stale/unlabeled; rebuilding from current source.")
    command = [
        "docker", "build", "--build-arg", "CODAMOSA_SOURCE_SHA=" + source_sha,
        "-t", CODAMOSA_IMAGE,
        "-f", str(ROOT / "codamosa" / "docker" / "Dockerfile"),
        str(ROOT / "codamosa"),
    ]
    try:
        result = subprocess.run(
            command, capture_output=True, text=True, encoding="utf-8", errors="replace"
        )
        if result.returncode:
            return False, (
                "BLOCKED: build image CODAMOSA gagal (exit "
                + str(result.returncode) + ").\n"
                + redact(((result.stdout or "") + "\n" + (result.stderr or ""))[-5000:])
                + "\nPeriksa koneksi registry, ruang disk, dan Dockerfile."
            )
        return True, "OK: image CODAMOSA berhasil dibangun."
    except Exception as exc:
        return False, "BLOCKED: build image CODAMOSA gagal.\n" + explain_error(exc)


if codamosa_ok:
    image_ok, image_reason = ensure_codamosa_image()
    print(image_reason)
    if not image_ok:
        codamosa_ok = False
        codamosa_reason += "\n" + image_reason
else:
    image_reason = "Dilewati karena preflight CODAMOSA gagal."


Menggunakan image yang sinkron dengan source: pynguin-runner:latest


## Menjalankan generator dan evaluasi

Test yang berhasil saja masuk ke `rows.csv`. Bila generator gagal, task dicatat
sebagai `BLOCKED` dengan alasan dan saran perbaikan di `status.json`.


In [ ]:
rows = []
statuses = []
RESUME_KEYS = set()
if RESUME:
    _active_rows, _active_statuses = _storage.ResultStore._read_active(OUT)
    _selected = {(ds, str(t)) for ds, tids in TASKS.items() for t in tids}
    # Keep ALL historical rows/statuses (e.g. gpt-5.6-sol) so analysis can
    # compare them, but only configured models may enter RESUME_KEYS/generator.
    rows = [r for r in _active_rows if (r.get('dataset'), str(r.get('task'))) in _selected]
    statuses = [x for x in _active_statuses
                if x.get('status') == 'OK'
                and (x.get('dataset'), str(x.get('task'))) in _selected]
    _retry_count = sum(
        1 for x in _active_statuses
        if x.get('status') != 'OK'
        and (x.get('dataset'), str(x.get('task'))) in _selected
    )
    print(f'RESUME: {_retry_count} BLOCKED/INVALID lama akan dicoba ulang.')
    _configured = {('CoverUp', m) for m in COVERUP_MODELS} | {
        ('CODAMOSA', m) for m in CODAMOSA_MODELS}
    for _item in statuses:
        if (_item.get('status') == 'OK'
                and (_item.get('method'), _item.get('model')) in _configured
                ):
            RESUME_KEYS.add((_item.get('method'), _item.get('model'),
                             _item.get('dataset'), str(_item.get('task'))))
    print(f'RESUME=1: loaded {len(rows)} rows and {len(statuses)} statuses; '
          f'{len(RESUME_KEYS)} configured-model OK keys.')


def blocked(method, dataset, task, reason):
    item = {
        "method": method,
        "dataset": dataset,
        "task": str(task),
        "status": "BLOCKED",
        "message": redact(reason),
    }
    statuses.append(item)
    print("\n[" + method + " | " + dataset + " " + str(task) + "] BLOCKED")
    print(item["message"])


def validate_generated_suite(method, layout, tests, work):
    validation_work = work / "validation"
    if validation_work.exists():
        shutil.rmtree(validation_work)
    validation_work.mkdir(parents=True)
    try:
        invalid = []
        assertion_total = 0
        for test_file in tests:
            valid, reason, assertions = em.suite_validity(Path(test_file))
            if not valid:
                invalid.append(Path(test_file).name + " (" + reason + ")")
            assertion_total += assertions
        if invalid:
            raise RuntimeError("INVALID_GENERATED_TEST: " + "; ".join(invalid))
        prepared = em.prepare_suite(layout, tests, validation_work)
        if assertion_total <= 0:
            raise RuntimeError("INVALID_GENERATED_TEST: no assertions can be evaluated")
        # CM tests import the dotted module (e.g. ``ansible.cli.adhoc``), so the
        # benchmark package root must be on sys.path exactly as coverage_of and
        # pass_rate_of already do.  A validation-only PYTHONPATH made every CM
        # suite fail collection with ModuleNotFoundError.
        _pkg_root = prepared.get("package_root") or ""
        _collect_pypath = (os.pathsep.join([_pkg_root, str(validation_work)])
                           if _pkg_root else str(validation_work))
        collect_env = dict(os.environ, PYTHONPATH=_collect_pypath, NO_COLOR="1",
                           PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
        collect_cmd = [
            em.PY, em.PYTEST, em.PYTEST_MODULE, "--collect-only", "-q", "-p", "no:cacheprovider",
            "--no-header", "-o", "addopts=", *[p.name for p in prepared["pytest_files"]]
        ]
        collect = subprocess.run(
            collect_cmd, cwd=str(validation_work), env=collect_env,
            capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=120,
        )
        (work / "pytest_collect.stdout.log").write_text(collect.stdout or "", encoding="utf-8")
        (work / "pytest_collect.stderr.log").write_text(collect.stderr or "", encoding="utf-8")
        if collect.returncode != 0 or not prepared["units"]:
            raise RuntimeError(
                "Suite " + method + " gagal import/pytest collection sebelum evaluasi. Inspect "
                + str(work / "pytest_collect.stderr.log")
                + "\n"
                + redact(((collect.stdout or "") + "\n" + (collect.stderr or ""))[-3000:])
            )
        return prepared
    finally:
        shutil.rmtree(validation_work, ignore_errors=True)


def _model_slug(model):
    return model.replace("/", "_")


def _run_coverup_task_with_model(dataset, task, model):
    layout = em.LAYOUTS[dataset](str(task))
    work = OUT / "work" / "coverup" / _model_slug(model) / dataset / str(task)
    llm_calls_log = (OUT / "llm_calls" / "CoverUp" / _model_slug(model)
                     / dataset / (str(task) + ".jsonl"))
    result = em.run_coverup(
        Path(layout["put"]), work / "tests", work,
        model=model, timeout=COVERUP_TIMEOUT,
        llm_calls_log=llm_calls_log,
        module_name=layout.get("module_name"),
        package_root=Path(layout["package_root"]) if layout.get("package_root") else None,
        extra=("--max-attempts", str(COVERUP_MAX_ATTEMPTS),
               "--model-temperature", str(COVERUP_MODEL_TEMPERATURE)),
    )
    if result is None or not result.ok:
        raise RuntimeError(
            (result.error if result else "CoverUp tidak dijalankan")
            or "CoverUp exited non-zero; inspect " + str(work / "coverup.log")
        )
    if not result.generated_tests:
        raise RuntimeError(
            "CoverUp selesai tetapi tidak menghasilkan test_coverup_*.py; inspect "
            + str(work / "coverup.log")
        )
    generated = [Path(p) for p in result.generated_tests]
    validate_generated_suite("CoverUp", layout, generated, work)
    row = em.evaluate(
        dataset, str(task), generated, "CoverUp",
        wall_time_s=result.wall_time_s, cost_usd=result.cost_usd,
        workroot=OUT / "work",
    )
    row["generator_ok"] = True
    row["requested_model"] = model
    row["model"] = BACKBONE_LABEL or model
    return row


def _run_codamosa_task_with_model(dataset, task, model):
    layout = em.LAYOUTS[dataset](str(task))
    source = Path(layout["put"]).resolve()
    module = layout.get("module_name") or source.stem
    work = OUT / "work" / "codamosa" / _model_slug(model) / dataset / str(task)
    project = Path(layout.get("package_root") or source.parent).resolve()
    output = work / "output"
    if work.exists():
        shutil.rmtree(work)
    output.mkdir(parents=True)
    package_txt = Path(layout.get("package_txt") or (project / "package.txt"))
    if not package_txt.is_file():
        raise FileNotFoundError("package.txt benchmark hilang: " + str(package_txt))

    base = os.getenv("CODAMOSA_BASE_URL", "https://router.octalabs.id/v1").strip().rstrip("/")
    relative = os.getenv("CODAMOSA_COMPLETIONS_PATH", "/chat/completions").strip()
    key = os.getenv("OCTA_API_KEY") or os.getenv("CODAMOSA_API_KEY") or os.getenv("OPENAI_API_KEY", "")
    args = [
        "--assertion-generation", "MUTATION_ANALYSIS",
        "--project_path", "/input",
        "--module-name", module,
        "--output-path", "/output",
        "--maximum_search_time", str(CODAMOSA_SEARCH_SECONDS),
        "--algorithm", "CODAMOSA",
        "--output_variables", "TargetModule,LineCoverage,LLMCalls,LLMQueryTime",
        "--report-dir", "/output",
        "--model_name", model,
        "--model_base_url", base,
        "--model_relative_url", relative,
        "--temperature", str(CODAMOSA_TEMPERATURE),
        "--large_language_model_seeding", "True",
        "--uninterpreted_statements", "ONLY",
        "--include-partially-parsable", "True",
        "--allow-expandable-cluster", "True",
        "--seeded_testcases_reuse_probability", "1",
        "--population", str(CODAMOSA_POPULATION),
    ]
    llm_calls_log = (OUT / "llm_calls" / "CODAMOSA" / _model_slug(model)
                     / dataset / (str(task) + ".jsonl")).resolve()
    llm_calls_log.parent.mkdir(parents=True, exist_ok=True)
    llm_calls_log.write_text("", encoding="utf-8")
    auth_file = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", prefix="codamosa-auth-", encoding="utf-8", delete=False
        ) as handle:
            auth_file = Path(handle.name)
            handle.write("--authorization-key\n" + key + "\n")
        auth_file.chmod(0o600)
        _safe_task = "".join(ch if ch.isalnum() else "-" for ch in str(task))
        _container_name = ("prpl-codamosa-" + _model_slug(model) + "-" +
                           "main-" + dataset.lower() + "-" + _safe_task)
        # Kill an exact-name leftover from a prior interrupted attempt before
        # starting. This is scoped; it never touches unrelated Docker work.
        subprocess.run(["docker", "rm", "-f", _container_name],
                       capture_output=True, text=True, timeout=60)
        command = [
            "docker", "run", "--rm", "--name", _container_name,
            "--label", "teori.experiment=codamosa",
                        "--label", "teori.dataset=" + dataset,
            "--label", "teori.task=" + str(task),
            "--cpus", str(CODAMOSA_DOCKER_CPUS),
            "-v", str(project) + ":/input:ro",
            "-v", str(output) + ":/output",
            "-v", str(project) + ":/package:ro",
            "-v", str(auth_file) + ":/run/codamosa-auth:ro",
            "-v", str(llm_calls_log) + ":/output/llm_calls.jsonl",
            "-e", "CODAMOSA_LLM_CALLS_LOG=/output/llm_calls.jsonl",
            "-e", "CODAMOSA_MAX_TOKENS=" + os.getenv("CODAMOSA_MAX_TOKENS", "2048"),
            "-e", "CODAMOSA_REQUEST_TIMEOUT=" + os.getenv("CODAMOSA_REQUEST_TIMEOUT", "600"),
            "-e", "CODAMOSA_SEND_STOP=" + os.getenv("CODAMOSA_SEND_STOP", "0"),
            # Pacing + empty-content recovery inside the container.
            "-e", "CODAMOSA_CALL_SPACING=" + os.getenv("CODAMOSA_CALL_SPACING", "1.2"),
            "-e", "CODAMOSA_EMPTY_CONTENT_RETRIES=" + os.getenv("CODAMOSA_EMPTY_CONTENT_RETRIES", "1"),
            "-e", "CODAMOSA_LLM_RETRIES=" + os.getenv("CODAMOSA_LLM_RETRIES", "2"),
            "-e", "RPM_GATE_STATE=/output/rpm_gate.json",
            "-v", str(OUT / "rpm_gate.json") + ":/output/rpm_gate.json",
            "-e", "RPM_LIMIT=" + os.getenv("RPM_LIMIT", "0"),
            "-v", str(ROOT / "src" / "rate_gate.py") + ":/pynguin/rate_gate.py:ro",
            CODAMOSA_IMAGE,
            "@/run/codamosa-auth",
        ] + args
        started = time.perf_counter()
        try:
            result = subprocess.run(
                command,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                timeout=(CODAMOSA_SEARCH_SECONDS
                         + max(0, int(os.getenv("CODAMOSA_CONTAINER_GRACE", "600")))),
            )
        except subprocess.TimeoutExpired as exc:
            subprocess.run(["docker", "rm", "-f", _container_name],
                           capture_output=True, text=True, timeout=60)
            (work / "docker.stdout.log").write_text(
                str(exc.stdout or ""), encoding="utf-8")
            (work / "docker.stderr.log").write_text(
                str(exc.stderr or ""), encoding="utf-8")
            raise RuntimeError(
                "CODAMOSA container timeout; container removed: " + _container_name + "\n"
                + redact((exc.stdout or "") + (exc.stderr or ""))
            ) from exc
    finally:
        if auth_file is not None:
            auth_file.unlink(missing_ok=True)
    elapsed = time.perf_counter() - started
    (work / "docker.stdout.log").write_text(result.stdout or "", encoding="utf-8")
    (work / "docker.stderr.log").write_text(result.stderr or "", encoding="utf-8")
    if result.returncode:
        _combined_log = redact(
            ((result.stdout or "") + "\n" + (result.stderr or ""))[-5000:])
        if result.returncode in (137, -9) or " Killed" in _combined_log:
            try:
                import psutil as _ps
                _ram_available = _ps.virtual_memory().available / (1024 ** 3)
            except Exception:
                _ram_available = None
            _ram_text = ("%.2f GB" % _ram_available
                         if _ram_available is not None else "tidak terdeteksi")
            raise RuntimeError(
                "CODAMOSA_OOM: Docker exit code 137/SIGKILL. Proses Pynguin "
                "dibunuh karena tekanan RAM host/WSL/Docker. RAM tersedia saat "
                "diagnosis: " + _ram_text + ". Turunkan TASK_WORKERS atau "
                "CONFIG_MAX_LOAD, lalu naikkan CODAMOSA_RAM_GB_PER_WORKER "
                "(default 3 GB).\n" + _combined_log)
        raise RuntimeError(
            "CODAMOSA/Docker exit code " + str(result.returncode) + ":\n"
            + _combined_log
        )
    generated = []
    invalid = []
    primary = [p for p in sorted(output.rglob("test_*.py"))
               if not p.name.endswith("_failing.py")
               and not p.name.endswith("_failing_no_metadata.py")]
    for candidate in primary:
        valid, reason, _assertions = em.suite_validity(candidate)
        if valid:
            generated.append(candidate)
        else:
            invalid.append(candidate.name + " (" + reason + ")")
    diagnostic = sorted(output.rglob("*_failing.py"))
    if not generated:
        detail = "INVALID_GENERATED_TEST: suite utama CODAMOSA kosong/tidak valid."
        if invalid:
            detail += " Artefak utama: " + ", ".join(invalid)
        if diagnostic:
            detail += " *_failing.py hanya diagnostik dan tidak dinilai sebagai sukses: "
            detail += ", ".join(p.name for p in diagnostic)
        # Persist every artefact the diagnosis needs so the failure can be
        # debugged from the notebook later without the remote machine.
        debug_dir = work / "debug_invalid_suite"
        debug_dir.mkdir(parents=True, exist_ok=True)
        for _art in sorted(list(output.rglob("*.py")) + list(output.rglob("statistics.csv"))):
            _rel = _art.relative_to(output)
            _dst = debug_dir / _rel
            _dst.parent.mkdir(parents=True, exist_ok=True)
            try:
                _dst.write_text(_art.read_text(encoding="utf-8", errors="replace"), encoding="utf-8")
            except OSError:
                pass
        for _log_name in ("docker.stdout.log", "docker.stderr.log"):
            _log = work / _log_name
            if _log.is_file():
                (debug_dir / _log_name).write_text(
                    _log.read_text(encoding="utf-8", errors="replace"), encoding="utf-8")
        detail += ". Debug artefacts: " + str(debug_dir)
        detail += ". Inspect " + str(work / "docker.stdout.log")
        raise RuntimeError(detail)

    validate_generated_suite("CODAMOSA", layout, generated, work)

    row = em.evaluate(
        dataset, str(task), generated, "CODAMOSA",
        wall_time_s=elapsed, workroot=OUT / "work",
    )
    # Token/call accounting from the runner's statistics.csv (needs
    # --output_variables). Providers that report usage add prompt/completion
    # tokens; otherwise only LLMCalls is available.
    # CODAMOSA writes variable names as column headers (single data row):
    #   TargetModule,LineCoverage,LLMCalls,LLMQueryTime,...
    # not a "name,value" format.  Map header->column, then read the data row.
    for _stat_csv in sorted(output.rglob("statistics.csv")):
        try:
            import csv as _csv
            with _stat_csv.open(encoding="utf-8", newline="") as _fh:
                _reader = _csv.reader(_fh)
                _header = next(_reader, None)
                if not _header:
                    continue
                _data = next(_reader, None)
                if not _data:
                    continue
                _col = {}
                for _idx, _name in enumerate(_header):
                    _col.setdefault(_name, _idx)
                for _name, _idx in _col.items():
                    if _idx >= len(_data):
                        continue
                    try:
                        _val = float(_data[_idx])
                    except (TypeError, ValueError):
                        continue
                    if _name == "LLMCalls":
                        row["calls_with_usage"] = int(_val)
                    elif _name == "prompt_tokens":
                        row["prompt_tokens"] = int(_val)
                    elif _name == "completion_tokens":
                        row["completion_tokens"] = int(_val)
        except (OSError, ValueError, StopIteration):
            pass

    row["generator_ok"] = True
    row["requested_model"] = model
    row["model"] = BACKBONE_LABEL or model
    return row


# Matrix: method x model x dataset x task. Each model has isolated outputs.
# Workers use isolated work directories and return data. Only the notebook's
# main thread mutates rows/statuses and writes checkpoints.
def _done(method, model, dataset, task):
    return (method, model, dataset, str(task)) in RESUME_KEYS


_TRANSIENT_ERROR_MARKERS = (
    "429", "rate limit", "too many requests", "502", "503", "504",
    "bad gateway", "service unavailable", "upstream service",
    "connectionerror", "connection error", "connection reset", "connection refused",
    "connection aborted", "network is unreachable", "temporary failure",
    "name resolution", "failed to resolve", "could not resolve", "dns",
    "timed out", "timeout", "docker daemon", "cannot connect to the docker",
    "no such host", "eof",
)


def _retryable_failure(exc, method):
    """Return whether an exception is safe to retry at the task boundary.

    Generated-suite validity is deterministic evidence, not a transient provider
    failure.  A transient HTTP/transport marker anywhere in the message is
    retryable -- including one preserved inside a CoverUp log excerpt, where the
    helper otherwise only sees "tidak menghasilkan test".

    CoverUp's *own* retry budget can still end in "giving up" after a transient
    connection error (e.g. ``InternalServerError: ... Connection error``).
    That is a provider hiccup, not a deterministic property of the task, so it
    stays retryable: the notebook then retries the whole invocation with a fresh
    connection.  Deterministic exhaustion ("Too many attempts") and
    no-coverage-progress outcomes carry no transient marker and fail fast
    instead of repeating a run that still costs another call.
    """
    text = str(exc).lower()
    if "invalid_generated_test" in text:
        return False
    return any(marker in text for marker in _TRANSIENT_ERROR_MARKERS)


def _run_with_retries(method, model, dataset, task, runner):
    last_exc = None
    for attempt in range(1, GENERATOR_ATTEMPTS + 1):
        try:
            return runner()
        except Exception as exc:  # noqa: BLE001 - per-task isolation boundary
            last_exc = exc
            retry = attempt < GENERATOR_ATTEMPTS and _retryable_failure(exc, method)
            if not retry:
                raise
            print(f"{method} {model} {dataset} {task}: attempt {attempt} transient failure; retry.")
            if GENERATOR_RETRY_SECONDS:
                time.sleep(GENERATOR_RETRY_SECONDS * attempt)
    raise last_exc  # pragma: no cover - loop always returns or raises above


def _blocked_record(method, dataset, task, reason):
    _reason = redact(reason)
    _status = "INVALID_GENERATED_TEST" if "INVALID_GENERATED_TEST" in _reason else "BLOCKED"
    _method, _sep, _model = method.partition(":")
    item = {"method": _method, "model": _model if _sep else None,
            "dataset": dataset, "task": str(task),
                "status": _status, "message": _reason}
    print("\n[" + method + " | " + dataset + " " + str(task) + "] " + _status)
    print(item["message"])
    return item


def _run_one_task(dataset, task):
    task_rows, task_statuses = [], []

    if not coverup_ok:
        for model in COVERUP_MODELS:
            if not _done("CoverUp", model, dataset, task):
                task_statuses.append(_blocked_record(
                    "CoverUp:" + model, dataset, task, coverup_reason))
        return task_rows, task_statuses
    for model in ACTIVE_COVERUP_MODELS:
        if _done("CoverUp", model, dataset, task):
            if VERBOSE_TASK_LOG:
                print(f"RESUME skip CoverUp {model} {dataset} {task}")
            continue
        try:
            if VERBOSE_TASK_LOG:
                print("CoverUp mencoba model:", model, "|", dataset, task)
            row = _run_with_retries(
                "CoverUp", model, dataset, task,
                lambda: _run_coverup_task_with_model(dataset, task, model),
            )
            task_rows.append(row)
            task_statuses.append({"method": "CoverUp", "model": model,
                "dataset": dataset, "task": str(task), "status": "OK",
                "message": "Generation and evaluation completed."})
            if VERBOSE_TASK_LOG:
                print("CoverUp OK", model, dataset, task)
        except Exception as exc:
            task_statuses.append(_blocked_record(
                "CoverUp:" + model, dataset, task, explain_error(exc)))

    for model in COVERUP_MODELS:
        if model in ACTIVE_COVERUP_MODELS or _done("CoverUp", model, dataset, task):
            continue
        reason = next((x for x in coverup_errors if x.startswith(model + ":")), coverup_reason)
        task_statuses.append(_blocked_record(
            "CoverUp:" + model, dataset, task,
            "BLOCKED: model gagal preflight. " + reason))

    if not codamosa_ok:
        for model in CODAMOSA_MODELS:
            if not _done("CODAMOSA", model, dataset, task):
                task_statuses.append(_blocked_record(
                    "CODAMOSA:" + model, dataset, task, codamosa_reason))
        return task_rows, task_statuses
    for model in ACTIVE_CODAMOSA_MODELS:
        if _done("CODAMOSA", model, dataset, task):
            if VERBOSE_TASK_LOG:
                print(f"RESUME skip CODAMOSA {model} {dataset} {task}")
            continue
        try:
            if VERBOSE_TASK_LOG:
                print("CODAMOSA mencoba model:", model, "|", dataset, task)
            row = _run_with_retries(
                "CODAMOSA", model, dataset, task,
                lambda: _run_codamosa_task_with_model(dataset, task, model),
            )
            task_rows.append(row)
            task_statuses.append({"method": "CODAMOSA", "model": model,
                "dataset": dataset, "task": str(task), "status": "OK",
                "message": "Generation and evaluation completed."})
            if VERBOSE_TASK_LOG:
                print("CODAMOSA OK", model, dataset, task)
        except Exception as exc:
            task_statuses.append(_blocked_record(
                "CODAMOSA:" + model, dataset, task, explain_error(exc)))

    for model in CODAMOSA_MODELS:
        if model in ACTIVE_CODAMOSA_MODELS or _done("CODAMOSA", model, dataset, task):
            continue
        reason = next((x for x in codamosa_errors if x.startswith(model + ":")), codamosa_reason)
        task_statuses.append(_blocked_record(
            "CODAMOSA:" + model, dataset, task,
            "BLOCKED: model gagal preflight. " + reason))

    return task_rows, task_statuses

def _adaptive_worker_count():
    """Scale task workers to live CPU/RAM, never above CONFIG_MAX_LOAD.

    The readiness cell decides the *initial* load.  Long runs drift (other
    apps close, Docker containers end), so this re-checks before the pool is
    created.  RAM is the binding constraint; use CODAMOSA_RAM_GB_PER_WORKER and keep
    2 GB for the OS/remote session. Requests are capped by the RPM gate.
    """
    try:
        import psutil as _ps
        ram_avail_gb = _ps.virtual_memory().available / (1024 ** 3)
    except Exception:
        ram_avail_gb = None
    cpu = os.cpu_count() or 1
    reserved = max(2, int(os.getenv("CPU_RESERVED_THREADS", "2")))
    max_load = max(1, int(os.getenv("CONFIG_MAX_LOAD", "2")))
    per_task_threads = 1 + max(1, EVAL_MUTANT_WORKERS)
    by_cpu = max(1, (cpu - reserved) // per_task_threads)
    by_ram = max_load if ram_avail_gb is None else max(
        1, int((ram_avail_gb - 2.0) // CODAMOSA_RAM_GB_PER_WORKER))
    target = max(1, min(by_cpu, by_ram, max_load, len(_task_jobs)))
    if not os.getenv("AUTO_SCALE", "0").strip().lower() in {"1", "true", "yes"}:
        target = max(1, min(TASK_WORKERS, len(_task_jobs)))
    print(f"Adaptive workers: {target} (cpu={cpu}, ram_avail={ram_avail_gb}, "
          f"by_cpu={by_cpu}, by_ram={by_ram}, cap={max_load})")
    return target


from concurrent.futures import ThreadPoolExecutor, as_completed


def _cleanup_codamosa_containers():
    """Remove only containers created by this experiment's CodaMOSA runner.

    ``docker run --rm`` normally cleans up, but interrupted notebook kernels can
    leave Created/Running containers behind.  Labels make the cleanup targeted
    so unrelated Docker workloads are never touched.
    """
    try:
        cp = subprocess.run(
            ["docker", "ps", "-aq", "--filter", "label=teori.experiment=codamosa"],
            capture_output=True, text=True, timeout=30,
        )
        ids = [x for x in cp.stdout.split() if x]
        if ids:
            subprocess.run(["docker", "rm", "-f", *ids], capture_output=True,
                           text=True, timeout=120)
            print(f"Cleaned {len(ids)} CodaMOSA experiment container(s).")
    except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
        print("Container cleanup warning:", exc)


# Remove leftovers from an earlier interrupted run before allocating resources.
_cleanup_codamosa_containers()

_task_jobs = [(dataset, str(task))
              for dataset, task_ids in TASKS.items()
              for task in task_ids]
_progress_started = time.monotonic()
_progress_done = 0
_progress_total = len(_task_jobs)
_progress_active_models = len(ACTIVE_COVERUP_MODELS) + len(ACTIVE_CODAMOSA_MODELS)
_progress_total_model_tasks = _progress_total * _progress_active_models
print("=" * 72)
print("GENERATOR + EVALUATION PLAN")
print("=" * 72)
print("Datasets:", ", ".join(f"{ds}={len(ids)} task" for ds, ids in TASKS.items()))
print("Active models: CoverUp=", len(ACTIVE_COVERUP_MODELS),
      "CODAMOSA=", len(ACTIVE_CODAMOSA_MODELS))
print("Task pipelines:", _progress_total,
      "| model-task jobs:", _progress_total_model_tasks,
      "| workers:", TASK_WORKERS)
print("ETA: dihitung dinamis setelah task pertama selesai; bukan waktu pasti karena API/retry/mutation dapat berubah.")
try:
    if _task_jobs:
        _worker_count = min(_adaptive_worker_count(), len(_task_jobs))
        print(f"Starting {_worker_count} parallel task workers for {len(_task_jobs)} tasks.")
        with ThreadPoolExecutor(max_workers=_worker_count,
                                thread_name_prefix="experiment-task") as _executor:
            _future_to_task = {
                _executor.submit(_run_one_task, dataset, task): (dataset, task)
                for dataset, task in _task_jobs
            }
            for _future in as_completed(_future_to_task):
                dataset, task = _future_to_task[_future]
                try:
                    _task_rows, _task_statuses = _future.result()
                except Exception as exc:
                    _task_rows, _task_statuses = [], []
                    _reason = "Unexpected task worker error: " + explain_error(exc)
                    for model in COVERUP_MODELS:
                        if not _done("CoverUp", model, dataset, task):
                            _task_statuses.append(_blocked_record(
                                "CoverUp:" + model, dataset, task, _reason))
                    for model in CODAMOSA_MODELS:
                        if not _done("CODAMOSA", model, dataset, task):
                            _task_statuses.append(_blocked_record(
                                "CODAMOSA:" + model, dataset, task, _reason))
                rows.extend(_task_rows)
                statuses.extend(_task_statuses)
                store.checkpoint(rows, statuses)
                _progress_done += 1
                _elapsed = time.monotonic() - _progress_started
                _avg_task = _elapsed / _progress_done if _progress_done else 0.0
                _remaining = max(0, _progress_total - _progress_done)
                _eta_s = _remaining * _avg_task
                _eta_h = int(_eta_s // 3600)
                _eta_m = int((_eta_s % 3600) // 60)
                _eta_sec = int(_eta_s % 60)
                _elapsed_m = int(_elapsed // 60)
                _elapsed_sec = int(_elapsed % 60)
                _ok_now = sum(x.get("status") == "OK" for x in statuses)
                _blocked_now = sum(x.get("status") in {"BLOCKED", "INVALID_GENERATED_TEST"} for x in statuses)
                _pct = (100.0 * _progress_done / _progress_total) if _progress_total else 100.0
                print(
                    f"[PROGRESS] {_progress_done}/{_progress_total} tasks ({_pct:.1f}%) | "
                    f"current={dataset}:{task} | rows={len(rows)} | "
                    f"OK={_ok_now} blocked/invalid={_blocked_now} | "
                    f"elapsed={_elapsed_m}m {_elapsed_sec:02d}s | "
                    f"ETA~{_eta_h}h {_eta_m:02d}m {_eta_sec:02d}s"
                )
    else:
        print("No tasks selected; nothing to run.")
finally:
    _cleanup_codamosa_containers()


RESUME: 0 BLOCKED/INVALID lama akan dicoba ulang.
RESUME=1: loaded 2 rows and 2 statuses; 2 configured-model OK keys.
GENERATOR + EVALUATION PLAN
Datasets: MT=163 task, CM=425 task
Active models: CoverUp= 1 CODAMOSA= 1
Task pipelines: 588 | model-task jobs: 1176 | workers: 1
ETA: dihitung dinamis setelah task pertama selesai; bukan waktu pasti karena API/retry/mutation dapat berubah.
Adaptive workers: 1 (cpu=16, ram_avail=3.6271438598632812, by_cpu=4, by_ram=1, cap=8)
Starting 1 parallel task workers for 588 tasks.
RESUME skip CoverUp openai/auto MT f001
RESUME skip CODAMOSA auto MT f001
CoverUp mencoba model: openai/auto | MT f002
[PROGRESS] 1/588 tasks (0.2%) | current=MT:f001 | rows=2 | OK=2 blocked/invalid=0 | elapsed=0m 00s | ETA~0h 00m 10s
CoverUp OK openai/auto MT f002
CODAMOSA mencoba model: auto | MT f002
CODAMOSA OK auto MT f002
CoverUp mencoba model: openai/auto | MT f003
[PROGRESS] 2/588 tasks (0.3%) | current=MT:f002 | rows=4 | OK=4 blocked/invalid=0 | elapsed=7m 47s | ETA

## Simpan hasil

`rows.csv` hanya berisi task yang selesai digenerate dan dievaluasi. `status.json`
berisi status semua task, termasuk alasan metode yang belum dapat berjalan.


In [ ]:
expected_statuses = sum(len(task_ids) for task_ids in TASKS.values()) * (len(COVERUP_MODELS) + len(CODAMOSA_MODELS))
manifest = store.finalize(rows, statuses, expected_statuses=expected_statuses)
print("Run manifest:", json.dumps(manifest, indent=2, ensure_ascii=False))
if not manifest["promoted"]:
    print("Active results preserved; inspect candidate:", store.candidate)

print("Rows:", OUT / "rows.csv")
print("Statuses:", OUT / "status.json")
print("OK:", sum(item["status"] == "OK" for item in statuses))
print("BLOCKED:", sum(item["status"] == "BLOCKED" for item in statuses))
print("INVALID_GENERATED_TEST:", sum(item["status"] == "INVALID_GENERATED_TEST" for item in statuses))

print("\nResult summary:")
print("OK:", sum(item["status"] == "OK" for item in statuses))
print("BLOCKED:", sum(item["status"] == "BLOCKED" for item in statuses))
print("INVALID_GENERATED_TEST:", sum(item["status"] == "INVALID_GENERATED_TEST" for item in statuses))
if rows:
    summary_groups = {}
    for row in rows:
        key = (row.get("dataset"), row.get("method"), row.get("model"))
        summary_groups.setdefault(key, []).append(row)
    for (dataset, method, model), group in sorted(summary_groups.items()):
        def avg(field):
            vals = [float(r[field]) for r in group if r.get(field) not in (None, "")]
            return round(sum(vals) / len(vals), 2) if vals else None
        print(
            f"{dataset} | {method} | {model}: "
            f"tasks={len(group)}, "
            f"coverage={avg('line_pct')}, "
            f"pass_rate={avg('pass_rate')}, "
            f"mutation_score={avg('mutation_score')}, "
            f"wall_time_s={avg('wall_time_s')}, "
            f"cost_usd={avg('cost_usd')}"
        )
else:
    print("No successful rows; inspect status.json for blocked methods/tasks.")



Run manifest: {
  "run_id": "20261019T214929Z_srbyte-auto-full-budget",
  "valid": true,
  "promoted": false,
  "backup": null,
  "expected_statuses": 0,
  "actual_statuses": 0,
  "old_quality": [
    2,
    100.0,
    2
  ],
  "new_quality": [
    0,
    -1.0,
    0
  ],
  "unmeasured_rows": 0,
  "trustworthy": false,
  "mutant_set_conflicts": {}
}
Active results preserved; inspect candidate: C:\Users\adamf\Documents\Kuliah\ITS\Semester 1\Topik Dalam Pengujian Perangkat Lunak\Project\experiment_results\runs\20261019T214929Z_srbyte-auto-full-budget
Rows: C:\Users\adamf\Documents\Kuliah\ITS\Semester 1\Topik Dalam Pengujian Perangkat Lunak\Project\experiment_results\rows.csv
Statuses: C:\Users\adamf\Documents\Kuliah\ITS\Semester 1\Topik Dalam Pengujian Perangkat Lunak\Project\experiment_results\status.json
OK: 0
BLOCKED: 0
INVALID_GENERATED_TEST: 0

Result summary:
OK: 0
BLOCKED: 0
INVALID_GENERATED_TEST: 0
No successful rows; inspect status.json for blocked methods/tasks.


## Visualisasi perbandingan dan analisis akurasi

Cell berikut membaca **hanya hasil run saat ini yang berstatus OK**, lalu membuat gambar `experiment_results/comparison.png`. Label menampilkan metode dan model sebenarnya. Prefix `openai/` pada CoverUp adalah transport LiteLLM, bukan model berbeda.

Interpretasi berfokus pada **mutation score** sebagai ukuran kualitas utama. Coverage/pass rate yang tinggi belum berarti test akurat apabila mutant tetap hidup. Analisis alasan bersifat evidence-based: jumlah assertion, mutant survived/broken/timeout, validitas suite, dan karakter prompt/task.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

ok_keys = {(x.get("method"), x.get("model"), x.get("dataset"), str(x.get("task")))
           for x in statuses if x.get("status") == "OK"}
valid_rows = [r for r in rows
              if (r.get("method"), r.get("model"), r.get("dataset"), str(r.get("task"))) in ok_keys]
invalid_rows = [x for x in statuses if x.get("status") != "OK"]


def _float(value):
    """Parse a CSV value to float; empty/None/unparsable -> NaN."""
    if value is None:
        return float("nan")
    if isinstance(value, str) and not value.strip():
        return float("nan")
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")


def _int(value):
    """Parse to int; NaN/None -> 0."""
    parsed = _float(value)
    return 0 if np.isnan(parsed) else int(parsed)


def _model_name(model):
    return str(model or "-").removeprefix("openai/")


if not valid_rows:
    display(Markdown("**Belum ada hasil valid untuk divisualisasikan.** Periksa `status.json`; "
                     "suite kosong atau yang menelan exception sengaja tidak diberi skor."))
else:
    # Field names as written by src/eval_metrics.py, split by role so that
    # "not reported" (NaN after coercion) is never silently treated as zero.
    score_fields = ["mutation_score", "line_pct", "branch_pct", "pass_rate"]
    count_fields = ["killed", "survived", "broken", "timeout", "problematic",
                    "n_mutants", "n_tests", "n_passed", "n_assertions_evaluated",
                    "mutant_set_total", "lines_covered", "lines_total"]
    cost_fields = ["cost_usd", "wall_time_s"]
    metric_fields = count_fields + score_fields + cost_fields

    records = []
    for row in valid_rows:
        record = {"dataset": row.get("dataset"), "method": row.get("method"),
                  "model": _model_name(row.get("model")),
                  "task": str(row.get("task")),
                  "mutant_set_id": row.get("mutant_set_id")}
        for field in metric_fields:
            record[field] = _float(row.get(field))
        # mutation_score is None when the PUT has no mutants at all
        # (n_mutants == 0). Keep that as NaN, but flag it so downstream
        # tables can exclude it instead of averaging it as 0.
        record["mutation_scored"] = not np.isnan(record["mutation_score"])
        record["no_mutants"] = record["n_mutants"] == 0
        records.append(record)
    frame = pd.DataFrame(records)
    keys = ["dataset", "method", "model"]

    # --- Data availability: never let a missing column look like a zero ----
    coverage = []
    for field in metric_fields:
        present = frame[field].notna()
        coverage.append({"field": field, "rows_with_value": int(present.sum()),
                         "rows_missing": int((~present).sum()),
                         "pct_present": round(100.0 * present.mean(), 1)})
    display(Markdown("### Ketersediaan data per field (dari %d baris valid)" % len(frame)))
    display(pd.DataFrame(coverage))
    # --- Summary: means over rows that actually report a value ------------
    # (pandas skips NaN in .mean()), plus explicit scored-task counts.
    scorable = {field: "mean" for field in score_fields + ["wall_time_s"]}
    # min_count=1: a group with no reported cost stays NaN, never a fake $0.
    scorable["cost_usd"] = lambda values: values.sum(min_count=1)
    scorable["mutant_set_total"] = "max"
    summary = frame.groupby(keys, dropna=False).agg(scorable)
    summary["tasks"] = frame.groupby(keys, dropna=False).size()
    summary["mutation_scored_tasks"] = frame.groupby(keys, dropna=False)["mutation_scored"].sum()
    summary["killed_total"] = frame.groupby(keys, dropna=False)["killed"].sum()
    summary["survived_total"] = frame.groupby(keys, dropna=False)["survived"].sum()
    summary["timeout_total"] = frame.groupby(keys, dropna=False)["timeout"].sum()
    summary["broken_total"] = frame.groupby(keys, dropna=False)["broken"].sum()
    summary["assertions_total"] = frame.groupby(keys, dropna=False)["n_assertions_evaluated"].sum()

    summary = summary.reset_index()
    _first = ["dataset", "method", "model", "tasks", "mutation_scored_tasks"]
    summary = summary[[c for c in _first if c in summary.columns]
                      + [c for c in summary.columns if c not in _first]]
    display(Markdown("### Ringkasan per dataset x metode x model"))
    display(Markdown("*Mutation score dihitung hanya atas task yang mutant-nya dapat dieksekusi "
                     "(`mutation_scored_tasks`); task tanpa mutant atau yang seluruh mutant-nya "
                     "broken/timeout ditandai `NaN`, bukan 0.*"))
    display(summary.rename(columns={
        "mutation_score": "mutation_score%", "line_pct": "line_cov%",
        "branch_pct": "branch_cov%", "pass_rate": "pass_rate%",
        "wall_time_s": "gen_time_s", "cost_usd": "cost_usd_total",
        "n_assertions_evaluated": "assertions"}))

    conflicts = []
    for (dataset, task), group in frame.groupby(["dataset", "task"], dropna=False):
        ids = {i for i in group["mutant_set_id"] if isinstance(i, str) and i}
        if len(ids) > 1:
            conflicts.append(f"{dataset}:{task} -> {sorted(ids)}")
    if conflicts:
        display(Markdown("- **Peringatan perbandingan tidak setara**: task berikut memakai "
                         "`mutant_set_id` berbeda, sehingga mutation score tidak dapat "
                         "dibandingkan langsung: " + "; ".join(conflicts)))
    elif frame["mutant_set_id"].isna().all():
        display(Markdown("- **Kesetaraan mutant set belum dapat diverifikasi**: baris berasal "
                         "dari run lama tanpa `mutant_set_id`."))
    else:
        display(Markdown("- **Pemeriksaan kesetaraan lolos**: setiap task memakai satu "
                         "`mutant_set_id` yang sama untuk semua metode/model."))
    # --- Charts ---------------------------------------------------------------
    # Metrik 0-100 mendapat fixed y-axis (0-105). Rows tanpa nilai (NaN) di-drop
    # per-metric agar tidak muncul sebagai bar "0.0" yang menyesatkan.
    chart_metrics = [
        ("mutation_score", "Mutation score (%)", True),
        ("line_pct", "Line coverage (%)", True),
        ("branch_pct", "Branch coverage (%)", True),
        ("pass_rate", "Pass rate valid (%)", True),
        ("wall_time_s", "Generation time (s)", False),
    ]
    datasets = sorted(frame["dataset"].dropna().unique())
    if datasets:
        fig, axes = plt.subplots(
            len(datasets), len(chart_metrics),
            figsize=(4.2 * len(chart_metrics), 4.0 * len(datasets)),
            squeeze=False)
        for row_index, dataset in enumerate(datasets):
            group = frame[frame["dataset"] == dataset]
            for column_index, (field, title, fixed_scale) in enumerate(chart_metrics):
                ax = axes[row_index][column_index]
                # dropna: NaN tidak punya makna kalau metrik tidak dilaporkan
                scored = group.dropna(subset=[field])
                series = scored.groupby(["method", "model"], dropna=False)[field].mean()
                if series.empty:
                    ax.set_title(f"{dataset} - {title}", fontsize=10)
                    ax.text(0.5, 0.5, "tidak ada data", ha="center", va="center",
                            transform=ax.transAxes, fontsize=9, color="#666666")
                    ax.set_xticks([])
                    ax.set_yticks([])
                    continue
                labels = [f"{method}\n{model}" for method, model in series.index]
                values = [0.0 if np.isnan(v) else float(v) for v in series.values]
                colors = ["#3478bf" if method == "CoverUp" else "#ed8b2f"
                          for method, _ in series.index]
                bars = ax.bar(labels, values, color=colors)
                ax.set_title(f"{dataset} - {title}", fontsize=10)
                ax.grid(axis="y", alpha=.25)
                ax.tick_params(axis="x", labelsize=8)
                if fixed_scale:
                    ax.set_ylim(0, 105)
                for bar, value in zip(bars, values):
                    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                            f"{value:.1f}", ha="center", va="bottom", fontsize=8)
        fig.suptitle("CoverUp vs CODAMOSA - per dataset, metode, dan model (suite valid)",
                     fontsize=14, fontweight="bold")
        fig.tight_layout()
        chart_path = OUT / "comparison.png"
        fig.savefig(chart_path, dpi=160, bbox_inches="tight")
        plt.show()
        display(Image(filename=str(chart_path)))
        print("Chart disimpan di", chart_path)

    # --- Analisis: mengapa mutation score rendah ---------------------------
    lines = ["### Mengapa akurasi dapat rendah pada prompt/task tertentu?"]
    for (dataset, method, model), group in frame.groupby(keys, dropna=False):
        # Hanya ranking task yang benar-benar punya mutant (mutation_score != NaN).
        # Task tanpa mutant (NaN) tidak ikut jadi kandidat "worst".
        scored = group.dropna(subset=["mutation_score"])
        if scored.empty:
            lines.append(f"- **{method} / {model}**, {dataset}: tidak ada task dengan mutant "
                         "valid, sehingga mutation score tidak dapat dilaporkan.")
            continue
        worst = scored.sort_values("mutation_score").iloc[0]
        score = worst["mutation_score"]
        survived, killed = _int(worst["survived"]), _int(worst["killed"])
        broken, timed_out = _int(worst["broken"]), _int(worst["timeout"])
        assertions = worst["n_assertions_evaluated"]
        assertions_text = "tidak tercatat" if np.isnan(assertions) else str(int(assertions))
        lines.append(
            f"- **{method} / {model}**, prompt `{dataset}:{worst['task']}`: mutation score "
            f"{score:.1f}% "
            f"({killed} killed, {survived} survived, {broken} broken, {timed_out} timeout; "
            f"{assertions_text} assertion).")
        if assertions_text == "tidak tercatat":
            lines[-1] += (" Baris ini berasal dari run lama tanpa penghitungan assertion; "
                          "jalankan ulang pilot agar buktinya lengkap.")
        elif survived and killed == 0:
            lines[-1] += (" Test lebih banyak mengonfirmasi contoh normal/coverage daripada "
                          "membedakan implementasi benar dari mutant; oracle dan edge case pada "
                          "prompt ini kurang kuat.")
        elif survived:
            lines[-1] += (" Sebagian mutant masih lolos, menunjukkan batas kondisi atau oracle "
                          "hasil belum lengkap.")
        else:
            lines[-1] += " Semua mutant valid terbunuh; oracle task ini cukup diskriminatif."
    _unscored = frame[~frame["mutation_scored"]]
    _no_mut = int(_unscored["no_mutants"].sum())
    _bad_mut = len(_unscored) - _no_mut
    if _no_mut:
        lines.append(f"- **{_no_mut} task tanpa mutant** (`n_mutants == 0`) tidak diberi "
                     "mutation score; task tersebut dikeluarkan dari rata-rata, bukan dihitung 0.")
    if _bad_mut:
        lines.append(f"- **{_bad_mut} task bermutant tapi seluruh mutant broken/timeout** "
                     "tidak dapat diskor (mutation score `NaN`), bukan dihitung 0; "
                     "periksa apakah test menyebabkan crash atau timeout saat mutasi.")
    if invalid_rows:
        lines.append(f"- **{len(invalid_rows)} suite/status tidak diskor** (misalnya suite "
                     "kosong atau `except BaseException: pass`), sehingga pass rate tidak tampak "
                     "bagus secara palsu.")
    lines.append("- Kesimpulan sebab dibatasi pada artefak yang terukur; perbedaan skor tidak "
                 "boleh dianggap semata-mata kelemahan model karena metode pencarian, prompt, "
                 "waktu, dan task juga memengaruhi hasil.")
    display(Markdown("\n".join(lines)))


**Belum ada hasil valid untuk divisualisasikan.** Periksa `status.json`; suite kosong atau yang menelan exception sengaja tidak diberi skor.

## Tabel log LLM per panggilan

Cell berikut membaca JSONL di `experiment_results/llm_calls/` lalu menampilkan:

1. **Tabel per panggilan**: waktu, prompt + prompt token, output + output token, latency, cost USD.
2. **Tabel agregat per task**: jumlah panggilan, total token, total latency, total cost.
3. **Tabel hasil per task** (`rows.csv`) termasuk kolom akuntansi LLM baru.

Prompt/output dipotong 160 karakter agar muat di layar; file JSONL tetap menyimpan teks lengkap.


In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

_llm_log_root = OUT / "llm_calls"
_records = []
if _llm_log_root.exists():
    for _path in sorted(_llm_log_root.rglob("*.jsonl")):
        # Struktur: experiment_results/llm_calls/<method>/<model>/<dataset>/<task>.jsonl
        try:
            _rel = _path.relative_to(_llm_log_root)
            _parts = _rel.parts
            _method, _dataset, _task = _parts[0], _parts[-2], _path.stem
            # Model may contain a slash (historical openai/gpt-* path).  New
            # logs use a slug, but both formats are accepted.
            _model = "/".join(_parts[1:-2]).replace("openai_", "openai/")
        except (ValueError, IndexError):
            continue
        with _path.open(encoding="utf-8") as _fh:
            for _line in _fh:
                _line = _line.strip()
                if not _line:
                    continue
                try:
                    _entry = json.loads(_line)
                except json.JSONDecodeError:
                    continue
                _records.append({
                    "method": _method,
                    "model": _entry.get("model") or _model,
                    "dataset": _dataset,
                    "task": _task,
                    "waktu": _entry.get("ts"),
                    "latency_s": _entry.get("latency_s"),
                    "prompt": str(_entry.get("prompt") or "").strip(),
                    "prompt_tokens": _entry.get("prompt_tokens"),
                    "output": str(_entry.get("output") or "").strip(),
                    "completion_tokens": _entry.get("completion_tokens"),
                    "total_tokens": _entry.get("total_tokens"),
                    "cost_usd": _entry.get("cost_usd"),
                    "finish_reason": _entry.get("finish_reason"),
                })

if not _records:
    print("Belum ada log per-panggilan di", _llm_log_root,
          "- jalankan ulang generator dulu agar tabel terisi.")
else:
    calls_df = pd.DataFrame(_records)
    _PREVIEW = 160
    calls_view = calls_df.copy()
    for _col in ("prompt", "output"):
        calls_view[_col] = (calls_view[_col].fillna("").astype(str)
                            .str.replace(r"\s+", " ", regex=True)
                            .str.slice(0, _PREVIEW))
        calls_view.loc[calls_view[_col].eq(""), _col] = "[tidak tersedia: log checkpoint agregat]"
    calls_view["latency_s"] = calls_view["latency_s"].fillna(
        "[tidak tersedia: log checkpoint agregat]")
    display(Markdown("### Log LLM per panggilan (%d panggilan)" % len(calls_df)))
    display(calls_view)

    # Agregat per task; cost_usd memakai min_count agar None tetap None
    # (bukan 0) bila provider tidak melaporkan biaya sama sekali.
    _agg = (calls_df.groupby(["method", "model", "dataset", "task"], dropna=False)
            .agg(llm_calls=("waktu", "size"),
                 prompt_tokens=("prompt_tokens", "sum"),
                 completion_tokens=("completion_tokens", "sum"),
                 total_tokens=("total_tokens", "sum"),
                 llm_wall_time_s=("latency_s", "sum"),
                 llm_cost_usd=("cost_usd", lambda s: s.sum(min_count=1)))
            .reset_index())
    _agg["llm_wall_time_s"] = _agg["llm_wall_time_s"].round(3)
    _agg["llm_cost_usd"] = _agg["llm_cost_usd"].round(6)
    display(Markdown("### Akuntansi LLM per task (agregat dari log per panggilan)"))
    display(_agg)

    _row_cols = ["dataset", "method", "model", "task", "line_pct", "pass_rate",
                 "mutation_score", "wall_time_s", "prompt_tokens",
                 "completion_tokens", "total_tokens", "cost_usd", "llm_calls",
                 "llm_output_chars", "llm_wall_time_s", "llm_calls_log"]
    if rows:
        _rows_df = pd.DataFrame(rows)
        _cols = [c for c in _row_cols if c in _rows_df.columns]
        display(Markdown("### Hasil per task (rows.csv)"))
        display(_rows_df[_cols])
    else:
        print("rows.csv belum punya baris sukses; lihat status.json.")


### Log LLM per panggilan (1170 panggilan)

,method,model,dataset,task,waktu,latency_s,prompt,prompt_tokens,output,completion_tokens,total_tokens,cost_usd,finish_reason
0,CODAMOSA,auto,HumanEval,0,2026-10-18T23:57:09Z,4.354,from typing import List def has_close_elements...,246,```python def test_has_close_elements(): asser...,763,1009,NaN,NaN
1,CODAMOSA,auto,HumanEval,0,2026-10-18T23:57:13Z,3.666,from typing import List def has_close_elements...,327,"assert has_close_elements([1.0, 2.0, 3.0], 0.5...",616,943,NaN,NaN
2,CODAMOSA,auto,HumanEval,0,2026-10-18T23:57:26Z,12.934,from typing import List def has_close_elements...,403,```python # Unit test for function has_close_e...,1007,1410,NaN,NaN
3,CODAMOSA,auto,HumanEval,0,2026-10-18T23:57:26Z,0.733,from typing import List def has_close_elements...,327,"assert has_close_elements([1.0, 2.0, 3.0], 0.5...",616,943,NaN,NaN
4,CODAMOSA,auto,HumanEval,0,2026-10-18T23:57:31Z,4.992,from typing import List def has_close_elements...,403,```python def test_has_close_elements(): # Exa...,916,1319,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1165,CoverUp,openai/auto,MT,pilot-fixed,2026-10-19T16:59:35Z,9.270,You are an expert Python test-driven developer...,888,"[tool_call] get_info({""name"": ""separate_paren_...",386,1274,NaN,tool_calls
1166,CoverUp,openai/auto,MT,pilot-fixed,2026-10-19T16:59:43Z,7.833,You are an expert Python test-driven developer...,1407,```python import pytest from typing import Lis...,520,1927,NaN,stop
1167,CoverUp,openai/auto,MT,pilot-fixed,2026-10-19T16:59:50Z,6.064,You are an expert Python test-driven developer...,1583,```python import pytest from typing import Lis...,497,2080,NaN,stop
1168,CoverUp,openai/auto,MT,pilot-fixed,2026-10-19T16:59:59Z,6.384,You are an expert Python test-driven developer...,1977,"[tool_call] get_info({""name"": ""f001.separate_p...",442,2419,NaN,tool_calls


### Akuntansi LLM per task (agregat dari log per panggilan)

,method,model,dataset,task,llm_calls,prompt_tokens,completion_tokens,total_tokens,llm_wall_time_s,llm_cost_usd
0,CODAMOSA,auto,HumanEval,0,32,10905,23727,34632,657.899,0.0
1,CODAMOSA,auto,HumanEval,1,60,26456,50894,77350,602.069,0.0
2,CODAMOSA,auto,HumanEval,10,136,24918,48412,73330,20.606,NaN
3,CODAMOSA,auto,HumanEval,2,24,10709,11553,22262,747.263,NaN
4,CODAMOSA,auto,HumanEval,3,40,17964,53267,71231,603.978,0.0
5,CODAMOSA,auto,HumanEval,4,100,23650,99373,123023,224.324,NaN
6,CODAMOSA,auto,HumanEval,5,106,40121,30870,70991,342.100,NaN
7,CODAMOSA,auto,HumanEval,6,4,2548,6668,9216,757.713,NaN
8,CODAMOSA,auto,HumanEval,7,104,34216,17980,52196,524.748,NaN
9,CODAMOSA,auto,HumanEval,8,104,38376,18564,56940,472.928,NaN


rows.csv belum punya baris sukses; lihat status.json.


## Tabel ringkas per metode (BB / RV / Flaky / LC / BC / MS)

| Kolom | Arti |
|---|---|
| **BB** | backbone (model LLM) yang dipakai metode ini. |
| **RV** | *raw validity*: proporsi tes usulan LLM yang lulus pada kode yang **belum** dimodifikasi/diperbaiki tool. `n/a` untuk Pynguin karena tesnya tidak dipilih oleh LLM. |
| **Flaky** | jumlah tes flaky di suite akhir. |
| **LC**, **BC** | line dan branch coverage. |
| **MS** | mutation score. |

LC, BC, dan MS dihitung pada tes valid. Pass rate suite akhir (mendekati 100% karena cara kerja tool) tidak diulang di sini; angka pass rate tetap dilaporkan di paket replikasi.

Nilai yang belum terukur ditandai `-` (bukan 0) supaya tidak terbaca sebagai hasil eksperimen.


In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

# --- Tabel ringkas per metode: BB / RV / Flaky / LC / BC / MS ----------------
# BB    = backbone (model LLM)
# RV    = raw validity: proporsi tes usulan LLM yang lulus pada kode ASLI
#         (sebelum diperbaiki tool); n/a untuk Pynguin (tes tidak dipilih LLM).
# Flaky = jumlah tes flaky di suite akhir.
# LC/BC = line / branch coverage; MS = mutation score (dihitung pada tes valid).
# Sel yang belum terukur ditampilkan '-' (bukan 0) agar tidak terbaca sebagai hasil.

MISSING = "-"


def _num(value):
    try:
        out = float(value)
    except (TypeError, ValueError):
        return None
    return None if pd.isna(out) else out


def _pct(value, digits=1):
    number = _num(value)
    return MISSING if number is None else "%.*f%%" % (digits, number)


def _counters_of(model):
    """Counter CoverUp dari checkpoint: G good, F failed, U useless, R retry.

    G = tes usulan yang lulus BERLAWANAN dengan kode asli (arah RV),
    F = tes yang gagal pada kode asli, R = retry akibat tes flaky.
    """
    slug = str(model or "").replace("/", "_")
    for base in (OUT / "work" / "coverup" / slug, OUT / "backups" / "coverup" / slug):
        if not base.exists():
            continue
        counters = {"G": 0, "F": 0, "U": 0, "R": 0}
        found = False
        for ckpt in sorted(base.rglob("coverup.checkpoint")):
            try:
                data = json.loads(ckpt.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError):
                continue
            raw = data.get("counters") or {}
            if not isinstance(raw, dict):
                continue
            found = True
            for key in counters:
                counters[key] += int(raw.get(key, 0) or 0)
        if found:
            return counters
    return None


if not rows:
    print("rows kosong; jalankan cell generator dulu agar tabel terisi.")
else:
    df = pd.DataFrame(rows)
    for column in ("line_pct", "branch_pct", "mutation_score", "pass_rate"):
        df[column] = (pd.to_numeric(df[column], errors="coerce")
                      if column in df.columns else pd.NA)

    table_rows = []
    for (dataset, method), group in df.groupby(["dataset", "method"], dropna=False):
        # LC/BC/MS hanya dihitung pada tes valid (semua unit lolos terisolasi).
        valid = group[group["pass_rate"].fillna(0) >= 100.0]
        if valid.empty:
            valid = group
        models = sorted({str(m).replace("openai/", "")
                         for m in group["model"].dropna().unique()})
        row = {
            "dataset": dataset,
            "metode": method,
            "BB": ", ".join(models) if models else MISSING,
            "n task": len(group),
            "RV": MISSING,
            "Flaky": MISSING,
            "LC": _pct(valid["line_pct"].mean()),
            "BC": _pct(valid["branch_pct"].mean()),
            "MS": _pct(valid["mutation_score"].mean()),
            "basis LC/BC/MS": "%d/%d task valid" % (len(valid), len(group)),
        }
        if str(method).lower() == "pynguin":
            row["RV"] = "n/a"
            row["Flaky"] = "n/a"
        elif "coverup" in str(method).lower():
            counters = _counters_of(group["model"].dropna().iloc[0])
            if counters and (counters["G"] + counters["F"]) > 0:
                total = counters["G"] + counters["F"]
                row["RV"] = _pct(100.0 * counters["G"] / total)
            if counters and counters["R"]:
                row["Flaky"] = str(counters["R"])
        table_rows.append(row)

    suite_table = pd.DataFrame(table_rows).sort_values(["dataset", "metode"])
    display(Markdown("### BB / RV / Flaky / LC / BC / MS (dihitung dari run saat ini)"))
    display(suite_table)
    display(Markdown(
        "- **RV** (*raw validity*) = proporsi tes usulan LLM yang lulus pada kode yang "
        "belum dimodifikasi/diperbaiki tool; `n/a` untuk Pynguin.\\n"
        "- **Flaky** = jumlah tes flaky di suite akhir (counter `R` CoverUp).\\n"
        "- **LC**, **BC**, **MS** dihitung pada tes valid; pass rate suite akhir "
        "dilaporkan di paket replikasi.\\n"
        "- Sel `-` berarti metrik belum bisa dihitung dari artefak run ini; itu bukan 0."))


rows kosong; jalankan cell generator dulu agar tabel terisi.


## Tabel V — Perbandingan pasangan

Cell berikut menjalankan uji **Wilcoxon signed-rank**, koreksi **Holm**, dan efek **Vargha–Delaney A12** untuk setiap pasangan metode + model.

- **M/S/K**: jumlah task tempat A lebih tinggi / sama / lebih rendah dari B.
- **A12 > 0,5**: A cenderung unggul; **A12 < 0,5**: B cenderung unggul.
- Pairing memakai irisan task yang mempunyai nilai valid pada kedua varian.
- `p-Holm` adalah p-value yang sudah dikoreksi, bukan p-value mentah.


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from IPython.display import display, Markdown
from itertools import combinations

# --- Tabel perbandingan pasangan: Wilcoxon signed-rank + Holm + A12 ----------
# Kolom: M/S/K (A>B, A==B, A<B), A12 (Vargha-Delaney), p-Holm.
# Holm dikoreksi atas semua uji (pasangan x metrik x dataset).
# Arah: A vs B -> A12 > 0.5 = A unggul; < 0.5 = B unggul.


def _a12(a: np.ndarray, b: np.ndarray) -> float:
    """Vargha-Delaney A12 = P(a > b) + 0.5 P(a == b)."""
    n, m = len(a), len(b)
    if n == 0 or m == 0:
        return float("nan")
    gt = sum(np.sum(a[i] > b) for i in range(n))
    eq = sum(np.sum(a[i] == b) for i in range(n))
    return (gt + 0.5 * eq) / (n * m)


def _holm(pvals: list) -> list:
    """Holm-Bonferroni correction (step-down, monotone)."""
    m = len(pvals)
    if m == 0:
        return []
    order = np.argsort(pvals)
    adj = [0.0] * m
    running = 0.0
    for rank, idx in enumerate(order):
        candidate = min(1.0, pvals[idx] * (m - rank))
        running = max(running, candidate)
        adj[idx] = running
    return adj


_METRIC_LABEL = {
    "line_pct": "line_pct",
    "branch_pct": "branch_pct",
    "mutation_score": "mutation_score",
    "pass_rate": "pass_rate",
}

if not rows:
    print("rows kosong; jalankan generator dulu.")
else:
    df = pd.DataFrame(rows)
    df["model"] = df.get("model", pd.Series("-", index=df.index)).fillna("-").astype(str)
    df["_variant"] = (df["method"].astype(str) + " [" +
                      df["model"].str.removeprefix("openai/") + "]")
    variants = sorted(df["_variant"].dropna().unique())
    datasets = sorted(df["dataset"].dropna().unique()) if "dataset" in df.columns else []

    _METRICS = []
    _skipped = []
    for _metric in ("line_pct", "branch_pct", "mutation_score", "pass_rate"):
        if _metric not in df.columns:
            _skipped.append((_metric, "kolom tidak ada di rows"))
            continue
        df[_metric] = pd.to_numeric(df[_metric], errors="coerce")
        if df[_metric].notna().sum() == 0:
            _skipped.append((_metric, "tidak ada nilai valid"))
            continue
        _METRICS.append(_metric)

    if _skipped:
        skip_lines = "; ".join(f"`{m}` ({r})" for m, r in _skipped)
        display(Markdown("**Metrik dilewati (tidak dilaporkan oleh run ini):** " + skip_lines
                         + ". Tidak ada baris dengan nilai valid, sehingga uji berpasangan "
                           "tidak dapat dijalankan untuk metrik tersebut."))

    if not _METRICS:
        print("Tidak ada metrik dengan nilai valid; jalankan generator dulu.")
    elif len(variants) < 2:
        print("Butuh minimal dua metode untuk uji berpasangan; ditemukan:", variants)
    else:
        pairs = list(combinations(variants, 2))  # (A, B) leksikografis
        test_rows = []   # satu baris per (dataset, A, B, metrik)
        raw_p = []       # p-value mentah untuk Holm

        for ds in datasets:
            sub = df[df["dataset"] == ds]
            for (mA, mB) in pairs:
                # groupby(task).last() also protects against an accidentally
                # duplicated checkpoint row within the same method+model.
                a_df = sub[sub["_variant"] == mA].groupby("task", as_index=True).last()
                b_df = sub[sub["_variant"] == mB].groupby("task", as_index=True).last()
                if a_df.empty or b_df.empty:
                    continue
                for met in _METRICS:
                    av = pd.to_numeric(a_df[met], errors="coerce").rename("A")
                    bv = pd.to_numeric(b_df[met], errors="coerce").rename("B")
                    paired = pd.concat([av, bv], axis=1, join="inner").dropna()
                    if len(paired) < 3:
                        continue
                    A = paired["A"].to_numpy(dtype=float)
                    B = paired["B"].to_numpy(dtype=float)
                    pair_idx = paired.index
                    diff = A - B
                    n_gt = int(np.sum(diff > 0))
                    n_eq = int(np.sum(diff == 0))
                    n_lt = int(np.sum(diff < 0))
                    a12_val = _a12(A, B)
                    nz = diff[diff != 0]
                    if len(nz) == 0:
                        p_val = 1.0
                    else:
                        try:
                            _, p_val = wilcoxon(A, B, alternative="two-sided")
                        except ValueError:
                            p_val = 1.0
                    test_rows.append({
                        "Dataset": ds,
                        "Pasangan (A vs B)": f"{mA} vs {mB}",
                        "Metrik": _METRIC_LABEL.get(met, met),
                        "n": len(pair_idx),
                        "M/S/K": f"{n_gt}/{n_eq}/{n_lt}",
                        "A12": round(a12_val, 3),
                        "p-Holm": p_val,
                        "_raw_p": p_val,
                    })
                    raw_p.append(p_val)

        if not test_rows:
            print("Tidak cukup data berpasangan untuk uji statistik "
                  "(butuh >= 3 task berpasangan per dataset x metrik).")
        else:
            holm = _holm(raw_p)
            for row, p_adj in zip(test_rows, holm):
                row["Signifikan"] = "\u2713" if p_adj < 0.05 else ""
                row["p-Holm"] = ("<0.0001*" if p_adj < 0.0001
                                 else f"{p_adj:.4f}" + ("*" if p_adj < 0.05 else ""))

            result = pd.DataFrame(test_rows).drop(columns=["_raw_p"])
            n_tests = len(result)
            display(Markdown(
                f"### Tabel V: Perbandingan Pasangan\n"
                f"*(uji Wilcoxon signed-rank, koreksi Holm atas {n_tests} uji)*\n\n"
                "M/S/K = jumlah unit tempat A lebih tinggi, setara, atau lebih rendah "
                "daripada B.\n"
                "A12 = peluang A mengungguli B (0,5 = tidak berbeda).\n"
                "p-value dikoreksi dengan metode Holm. Metrik yang tidak dilaporkan oleh "
                "run ini dilewati; perbandingan untuk metrik lainnya ada di paket replikasi."
            ))
            display(result)


rows kosong; jalankan generator dulu.
